# Stage 02_dynamics_feedback: dynamics_feedback
Configuration, inputs and results remain in the corresponding stages directory.


### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
dynamics_feedback_experiment_dir = ROOT / 'experiment/research/instability_guided_bp/stages/02_dynamics_feedback'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

dynamics_feedback_data_paths = ExperimentPaths(ROOT, dynamics_feedback_experiment_dir / 'config/paths.json')


In [ ]:
import json
import sys
from collections import defaultdict
from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
from time import perf_counter
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr, wilcoxon
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from beliefmatching import detector_error_model_to_check_matrices
from codes.ca_code import CA_9_2_5, TriangularCode
from ldpc import BpDecoder
from noises_ca import CircuitBuilder, ErrorRate

dynamics_feedback_config = json.loads((dynamics_feedback_experiment_dir / 'config/next_instability_guided_bp.json').read_text(encoding='utf-8'))

from experiment.utils.bp_trace import check_bp_cache
check_bp_cache(dynamics_feedback_data_paths, dynamics_feedback_config)

# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.
# Expected relationship: long_iterations >= benchmark_iterations >= diagnostic_iterations >=
# probe_iterations >= 2.
# primary_candidate_test_iterations: chosen from candidate_test_iterations.
# candidate_test_iterations: each item is within the bounds 2 <= value <= probe_iterations.
# late_fraction: in (0, 1].
# message_zero_tolerance: non-negative.
# native_validation_tolerance: positive.
# ms_scaling_factor: finite and positive.
# stability_threshold: positive.
# primary_top_k: chosen from top_k_values.
# top_k_values: each item is positive.
# Expected relationship: number of entries in composite_weights == 3.
# sum of composite_weights: approximately 1.
# composite_weights: each item is non-negative.
# primary_max_row_weight: chosen from candidate_max_row_weights.
# candidate_screen_limit: positive.
# samples_per_outcome: positive.
# random_support_repetitions: positive.
# targeting_random_seeds: positive.
# cross_distance_levels: values exactly {1, 2, 3}.
# cross_distance_error_weights: values exactly {2, 3}.
# bootstrap_repetitions: positive.
# cv_folds: at least 2.


In [ ]:
from experiment.utils import graph_intervention

dynamics_feedback_GraphContext = graph_intervention.GraphContext

from experiment.utils import bp_message_dynamics

dynamics_feedback_MessageTrace = bp_message_dynamics.MessageTrace

def dynamics_feedback_graph_context(
    p,
    H,
    L,
    priors,
):
    return graph_intervention.graph_context(p, H, L, priors, GraphContext=dynamics_feedback_GraphContext)

dynamics_feedback_context_cache = {}

def dynamics_feedback_context_for(level, p):
    return graph_intervention.context_for(
        level,
        p,
        CA_9_2_5=CA_9_2_5,
        CONTEXT_CACHE=dynamics_feedback_context_cache,
        CircuitBuilder=CircuitBuilder,
        ErrorRate=ErrorRate,
        TriangularCode=TriangularCode,
        detector_error_model_to_check_matrices=detector_error_model_to_check_matrices,
        graph_context=dynamics_feedback_graph_context,
    )

def dynamics_feedback_augmented_context(context, extra_rows):
    return graph_intervention.augmented_context(context, extra_rows, graph_context=dynamics_feedback_graph_context)

from experiment.utils import misc

dynamics_feedback_bit_array = misc.bit_array_variant_2
dynamics_feedback_bit_text = misc.bit_text_variant_2

def dynamics_feedback_logical_label(context, correction):
    return graph_intervention.logical_label(context, correction, bit_text=dynamics_feedback_bit_text)

def dynamics_feedback_trace_min_sum(
    context,
    syndrome,
    iterations,
    edge_damping=None,
    edge_scaling=None,
):
    return graph_intervention.trace_min_sum(context, syndrome, iterations, edge_damping, edge_scaling, CONFIG=dynamics_feedback_config, MessageTrace=dynamics_feedback_MessageTrace)

dynamics_feedback_percentile_rank = graph_intervention.percentile_rank

def dynamics_feedback_outcome_of(
    context,
    syndrome,
    truth,
    correction,
):
    return graph_intervention.outcome_of(context, syndrome, truth, correction, logical_label=dynamics_feedback_logical_label)

dynamics_feedback_top_region = graph_intervention.top_region
dynamics_feedback_conditional_outcome_rates = graph_intervention.conditional_outcome_rates

def dynamics_feedback_edge_maps(
    context,
    variable_scores,
    low,
    high,
    mode,
    rng=None,
):
    return graph_intervention.edge_maps(context, variable_scores, low, high, mode, rng, percentile_rank=dynamics_feedback_percentile_rank)

dynamics_feedback_CandidateRow = graph_intervention.CandidateRow

def dynamics_feedback_generate_redundant_candidates(H, max_row_weight):
    return graph_intervention.generate_redundant_candidates(H, max_row_weight, CandidateRow=dynamics_feedback_CandidateRow, bit_text=dynamics_feedback_bit_text)

dynamics_feedback_augmented_syndrome = graph_intervention.augmented_syndrome
dynamics_feedback_candidate_overlap = graph_intervention.candidate_overlap
dynamics_feedback_dtype = {
    'p': float,
    'err': str,
    'det': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}


In [ ]:
dynamics_feedback_data = dynamics_feedback_data_paths.run()
dynamics_feedback_input = dynamics_feedback_data_paths.input('error_weight')
dynamics_feedback_trace = dynamics_feedback_data_paths.input('trace')


# Next-stage instability-guided BP experiments

It uses an independent
configuration and output tree. Rank quantities are used only for localization;
all intervention-response gains use raw normalized quantities on the same fixed
variable region and the same iteration window.


In [ ]:
import math
from ldpc import BpOsdDecoder
from ldpc_modified import AdaptiveBpDecoder
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, cross_val_predict

NEXT_CONFIG = json.loads((dynamics_feedback_experiment_dir / 'config/next_config.json').read_text(encoding='utf-8'))
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.
# Expected relationship: probe_iterations >= maximum of candidate_test_iterations >= 3.
# primary_candidate_test_iterations: chosen from candidate_test_iterations.
# primary_top_k: positive.
# candidate_max_row_weight: at least 2.
# Expected relationship: candidate_screen_limit >= maximum of candidate_budget_values > 0.
# unbiased_candidate_limit: positive.
# samples_per_outcome: positive.
# random_control_seeds: positive.
# development_fraction: in (0, 1).
# hybrid_alphas: each item is in [0, 1].
# early_probe_iterations: each item is at least 2.
# Expected relationship: iterative_max_rounds >= plateau_patience > 0.
# plateau_delta_values: each item is non-negative.
# gating_quantiles: each item is in [0, 1].
# Expected relationship: 0 <= damping_gamma_min <= damping_gamma_max < 1.
# cross_distance_levels: [1, 2, 3].
# cross_distance_error_weights: [2, 3].
# stochastic_shots_per_point: positive.
# stochastic_batch_size: positive.
assert NEXT_CONFIG['longer_bp_iterations'] > dynamics_feedback_config['benchmark_iterations']
# Expected configuration values:
# native_validation_tolerance: positive.
NEXT_OUTPUT = dynamics_feedback_data_paths.run()

from experiment.utils.bp_trace import check_bp_cache
check_bp_cache(dynamics_feedback_data_paths, NEXT_CONFIG)

NEXT_STATS = dynamics_feedback_data_paths.output('stats')
NEXT_TRACE = dynamics_feedback_data_paths.output('trace')
NEXT_STATS.mkdir(parents=True, exist_ok=True)
NEXT_TRACE.mkdir(parents=True, exist_ok=True)

def next_log(level, payload):
    if NEXT_CONFIG['print_level'] >= level:
        print(payload)

next_log(2, {'next_stats': str(NEXT_STATS), 'next_trace': str(NEXT_TRACE)})


In [ ]:
@dataclass(frozen=True)
class RawDynamics:
    raw_trajectory: np.ndarray
    raw_peak: np.ndarray
    sign_flip: np.ndarray
    period_2: np.ndarray
    belief_instability: np.ndarray
    extrinsic_conflict: np.ndarray
    rank_trajectory: np.ndarray
    per_iteration: np.ndarray

RAW_METRICS = ('raw_trajectory', 'raw_peak', 'sign_flip', 'period_2', 'belief_instability', 'extrinsic_conflict')

def raw_dynamics(context, trace, iterations):
    assert 2 <= iterations < len(trace.c2v)
    eps = np.finfo(float).eps
    messages = trace.c2v[1:iterations + 1]
    edge_scale = np.abs(context.prior_llr[context.edge_variables]) + eps
    normalized = np.abs(np.diff(trace.c2v[:iterations + 1], axis=0)) / edge_scale
    signs = np.where(np.abs(messages) <= dynamics_feedback_config['message_zero_tolerance'], 0, np.where(messages > 0, 1, -1))
    flips = (signs[1:] != signs[:-1]).astype(float)
    belief_delta = np.abs(np.diff(trace.llr[:iterations + 1], axis=0))
    belief_delta /= np.abs(context.prior_llr)[None, :] + eps
    n = context.H.shape[1]
    trajectory = np.zeros(n)
    peak = np.zeros(n)
    sign_flip = np.zeros(n)
    period_2 = np.zeros(n)
    conflict = np.zeros(n)
    per_iteration = np.zeros((iterations, n))

    for variable, edge_ids in enumerate(context.variable_edges):
        if not len(edge_ids):
            continue
        per_iteration[:, variable] = normalized[:, edge_ids].mean(axis=1)
        trajectory[variable] = per_iteration[:, variable].mean()
        peak[variable] = per_iteration[:, variable].max()
        sign_flip[variable] = flips[:, edge_ids].mean() if len(flips) else 0.0

        if iterations >= 3:
            d1 = np.abs(np.diff(messages[:, edge_ids], axis=0)).mean(axis=0)
            d2 = np.abs(messages[2:, edge_ids] - messages[:-2, edge_ids]).mean(axis=0)
            period_2[variable] = np.mean(d1 * d1 / (d1 + d2 + eps))
        incoming = messages[:, edge_ids]
        conflict[variable] = np.mean(1.0 - np.abs(incoming.sum(axis=1)) / (np.abs(incoming).sum(axis=1) + eps))

    result = RawDynamics(trajectory, peak, sign_flip, period_2, belief_delta.mean(axis=0), conflict, dynamics_feedback_percentile_rank(trajectory), per_iteration)
    for name in RAW_METRICS:
        values = np.asarray(getattr(result, name))
        assert values.shape == (n,) and np.isfinite(values).all()

    return result

def metric_region_mean(metrics, name, region):
    values = np.asarray(getattr(metrics, name), dtype=float)
    region = np.asarray(region, dtype=int)
    assert region.size and np.all((0 <= region) & (region < len(values)))

    return float(values[region].mean())

def syndrome_residual_weights(context, syndrome, trace):
    syndrome = np.asarray(syndrome, dtype=np.uint8)[:context.H.shape[0]]
    return np.count_nonzero((context.H @ trace.hard.T).T % 2 ^ syndrome[None, :], axis=1)

def hard_decision_movement(trace):
    return np.count_nonzero(np.diff(trace.hard.astype(np.int8), axis=0), axis=1)

def outcome_at_first_valid(
    base_context,
    base_syndrome,
    truth,
    trace,
):
    index = trace.first_valid_iteration
    if index is None:
        index = len(trace.hard) - 1
    correction = trace.hard[index]
    valid, correct = dynamics_feedback_outcome_of(base_context, base_syndrome, truth, correction)

    return (index, valid, correct, correction)

def probe_raw_candidate(
    context,
    syndrome,
    candidate,
    region,
    baseline_by_window,
    baseline_trace,
):
    augmented = dynamics_feedback_augmented_context(context, candidate.row)
    augmented_s = dynamics_feedback_augmented_syndrome(syndrome, candidate.source_rows)
    probe_rows = []

    for iterations in NEXT_CONFIG['candidate_test_iterations']:
        started = perf_counter()
        trace = dynamics_feedback_trace_min_sum(augmented, augmented_s, iterations)
        probe_runtime = perf_counter() - started
        after = raw_dynamics(augmented, trace, iterations)
        record = {
            'test_iterations': iterations,
            'short_probe_syndrome_consistent': int(trace.syndrome_consistent[-1]),
        }

        for metric in RAW_METRICS:
            before_value = metric_region_mean(baseline_by_window[iterations], metric, region)
            after_value = metric_region_mean(after, metric, region)
            record[f'{metric}_before'] = before_value
            record[f'{metric}_after'] = after_value
            record[f'{metric}_gain'] = before_value - after_value

        rank_before = float(baseline_by_window[iterations].rank_trajectory[region].mean())
        rank_after = float(after.rank_trajectory[region].mean())
        record.update({
            'rank_before': rank_before,
            'rank_after': rank_after,
            'rank_gain': rank_before - rank_after,
        })
        before_w = syndrome_residual_weights(context, syndrome, baseline_trace)[:iterations + 1]
        after_w = syndrome_residual_weights(augmented, augmented_s, trace)
        record.update({
            'syndrome_weight_before': int(before_w[-1]),
            'syndrome_weight_after': int(after_w[-1]),
            'syndrome_progress_gain': int(before_w[-1] - after_w[-1]),
            'minimum_syndrome_weight_after': int(after_w.min()),
            'mean_hard_movement_after': float(hard_decision_movement(trace).mean()),
            'probe_runtime_seconds': probe_runtime,
            'probe_edge_updates': int(iterations * augmented.H.sum()),
        })
        probe_rows.append(record)

    return probe_rows

def full_raw_candidate(
    context,
    syndrome,
    truth,
    candidate,
    region,
    full_iterations=None,
):
    if full_iterations is None:
        full_iterations = dynamics_feedback_config['benchmark_iterations']
    augmented = dynamics_feedback_augmented_context(context, candidate.row)
    augmented_s = dynamics_feedback_augmented_syndrome(syndrome, candidate.source_rows)
    started = perf_counter()
    full_trace = dynamics_feedback_trace_min_sum(augmented, augmented_s, full_iterations)
    full_runtime = perf_counter() - started
    index, valid, correct, _ = outcome_at_first_valid(context, syndrome, truth, full_trace)
    final_metrics = raw_dynamics(augmented, full_trace, full_iterations)
    full = {
        'syndrome_consistent': int(valid),
        'logical_correct': int(correct),
        'iterations': int(index),
        'executed_iterations': int(full_iterations),
        'full_runtime_seconds': full_runtime,
        'full_edge_updates': int(full_iterations * augmented.H.sum()),
        'final_raw_trajectory_instability': metric_region_mean(final_metrics, 'raw_trajectory', region),
    }

    return full

def evaluate_raw_candidate(
    context,
    syndrome,
    truth,
    candidate,
    region,
    baseline_by_window,
    baseline_trace,
    full_iterations=None,
):
    probes = probe_raw_candidate(context, syndrome, candidate, region, baseline_by_window, baseline_trace)
    full = full_raw_candidate(context, syndrome, truth, candidate, region, full_iterations)
    return (probes, full)

def balanced_candidate_screen(
    candidates,
    overlap_scores,
    limit,
    seed,
):
    if len(candidates) <= limit:
        return list(candidates)
    order = np.lexsort((np.arange(len(candidates)), -np.asarray(overlap_scores)))
    top_count = limit // 3
    bottom_count = limit // 3
    chosen = list(order[:top_count]) + list(order[-bottom_count:])
    remaining = np.setdiff1d(np.arange(len(candidates)), chosen)
    rng = np.random.default_rng(seed)
    chosen.extend(rng.choice(remaining, size=limit - len(chosen), replace=False).tolist())
    assert len(set(chosen)) == limit

    return [candidates[index] for index in chosen]

def minmax_within_group(values):
    values = np.asarray(values, dtype=float)
    span = values.max() - values.min()
    return np.zeros_like(values) if span == 0 else (values - values.min()) / span

def stratified_development_ids(cases):
    rng = np.random.default_rng(NEXT_CONFIG['next_random_seed'])
    selected = set()
    for _, group in cases.groupby('failure_type', sort=True):
        ids = group.sample(frac=1, random_state=int(rng.integers(2 ** 31))).sample_id.to_numpy()
        count = max(1, min(len(ids) - 1, int(round(len(ids) * NEXT_CONFIG['development_fraction']))))
        selected.update(ids[:count].tolist())

    return selected

def grouped_binary_score(frame, features, target_name):
    target = frame[target_name].to_numpy(dtype=int)
    groups = frame.sample_id.to_numpy()
    if len(np.unique(target)) < 2 or frame.sample_id.nunique() < 2:
        return (np.nan, np.full(len(frame), np.nan))
    folds = min(dynamics_feedback_config['cv_folds'], frame.sample_id.nunique())
    splitter = GroupKFold(n_splits=folds)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

    probabilities = cross_val_predict(model, frame[features].to_numpy(), target, groups=groups, cv=splitter, method='predict_proba')[:, 1]
    return (roc_auc_score(target, probabilities), probabilities)


In [ ]:
dynamics_feedback_population_cases = pd.read_csv(dynamics_feedback_data_paths.input('graph_intervention_population_cases_csv'), dtype={
    'error': str,
    'syndrome': str,
    'true_sector': str,
    'residual': str,
})
assert dynamics_feedback_population_cases.sample_id.is_unique
cohort_parts = []

for dynamics_feedback_failure_type, dynamics_feedback_group in dynamics_feedback_population_cases.groupby('failure_type', sort=True):
    cohort_parts.append(dynamics_feedback_group.sample(n=min(len(dynamics_feedback_group), NEXT_CONFIG['samples_per_outcome']), random_state=NEXT_CONFIG['next_random_seed']))
next_cases = pd.concat(cohort_parts, ignore_index=True).sort_values('sample_id').reset_index(drop=True)
development_ids = stratified_development_ids(next_cases)
next_cases['split'] = np.where(next_cases.sample_id.isin(development_ids), 'development', 'test')

next_cases.groupby(['failure_type', 'split'], as_index=False).agg(cases=('sample_id', 'size')).to_csv(dynamics_feedback_data_paths.output('next_instability_input_audit_csv'), index=False)
next_cases.groupby(['failure_type', 'split'], as_index=False).size()


## Experiment A — Raw instability reduction vs repair

Test raw before/after probe changes against eventual repair. Unlike independently reranked percentiles, raw measurements retain a common scale for the intervention response.


In [ ]:
raw_candidate_rows = []
for dynamics_feedback_sample in tqdm(next_cases.itertuples(index=False), total=len(next_cases), desc='Experiment A: full candidate pool', disable=NEXT_CONFIG['print_level'] == 1):
    dynamics_feedback_, dynamics_feedback_context = dynamics_feedback_context_for(2, dynamics_feedback_sample.p)
    dynamics_feedback_syndrome = dynamics_feedback_bit_array(dynamics_feedback_sample.syndrome)
    localization_started = perf_counter()
    dynamics_feedback_baseline_trace = dynamics_feedback_trace_min_sum(dynamics_feedback_context, dynamics_feedback_syndrome, NEXT_CONFIG['probe_iterations'])
    localization_runtime = perf_counter() - localization_started
    localization_edge_updates = NEXT_CONFIG['probe_iterations'] * int(dynamics_feedback_context.H.sum())
    initial = raw_dynamics(dynamics_feedback_context, dynamics_feedback_baseline_trace, NEXT_CONFIG['probe_iterations'])
    dynamics_feedback_region = dynamics_feedback_top_region(initial.rank_trajectory, min(NEXT_CONFIG['primary_top_k'], dynamics_feedback_context.H.shape[1]))
    baseline_by_window = {iterations: raw_dynamics(dynamics_feedback_context, dynamics_feedback_baseline_trace, iterations) for iterations in NEXT_CONFIG['candidate_test_iterations']}
    dynamics_feedback_candidates = dynamics_feedback_generate_redundant_candidates(dynamics_feedback_context.H, NEXT_CONFIG['candidate_max_row_weight'])
    assert dynamics_feedback_candidates

    for dynamics_feedback_candidate in dynamics_feedback_candidates:
        dynamics_feedback_overlap_count, overlap_score = dynamics_feedback_candidate_overlap(dynamics_feedback_candidate, initial.rank_trajectory, dynamics_feedback_region)
        conflict_overlap = float(initial.extrinsic_conflict[np.flatnonzero(dynamics_feedback_candidate.row)].sum())
        dynamics_feedback_probes, dynamics_feedback_full = evaluate_raw_candidate(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_sample.true_sector, dynamics_feedback_candidate, dynamics_feedback_region, baseline_by_window, dynamics_feedback_baseline_trace)
        dynamics_feedback_success = bool(dynamics_feedback_full['syndrome_consistent'] and dynamics_feedback_full['logical_correct'])
        dynamics_feedback_baseline_success = dynamics_feedback_sample.failure_type == 'not_failure'

        for dynamics_feedback_probe in dynamics_feedback_probes:
            raw_candidate_rows.append({
                'sample_id': dynamics_feedback_sample.sample_id,
                'p': dynamics_feedback_sample.p,
                'failure_type': dynamics_feedback_sample.failure_type,
                'split': dynamics_feedback_sample.split,
                'candidate_id': dynamics_feedback_candidate.candidate_id,
                'source_rows': json.dumps(dynamics_feedback_candidate.source_rows),
                'row_weight': dynamics_feedback_candidate.row_weight,
                'row_support': json.dumps(np.flatnonzero(dynamics_feedback_candidate.row).tolist()),
                'region': json.dumps(dynamics_feedback_region.tolist()),
                'overlap_count': dynamics_feedback_overlap_count,
                'weighted_instability_overlap': overlap_score,
                'conflict_overlap': conflict_overlap,
                'localization_edge_updates': localization_edge_updates,
                'localization_runtime_seconds': localization_runtime,
                **dynamics_feedback_probe,
                **dynamics_feedback_full,
                'repair': int(not dynamics_feedback_baseline_success and dynamics_feedback_success),
                'regression': int(dynamics_feedback_baseline_success and (not dynamics_feedback_success)),
            })

raw_candidate_trials = pd.DataFrame(raw_candidate_rows)
assert raw_candidate_trials.groupby(['sample_id', 'candidate_id', 'test_iterations']).size().eq(1).all()
raw_candidate_trials.to_csv(dynamics_feedback_data_paths.output('raw_candidate_trials_csv'), index=False)
raw_primary = raw_candidate_trials[raw_candidate_trials.test_iterations == NEXT_CONFIG['primary_candidate_test_iterations']].copy()
raw_gain_rows = []
bootstrap_rng = np.random.default_rng(NEXT_CONFIG['next_random_seed'])
failure_candidates = raw_primary[raw_primary.failure_type != 'not_failure']

for dynamics_feedback_metric in RAW_METRICS:
    dynamics_feedback_gain = f'{dynamics_feedback_metric}_gain'
    dynamics_feedback_eligible = failure_candidates.dropna(subset=[dynamics_feedback_gain])
    auroc = roc_auc_score(dynamics_feedback_eligible.repair, dynamics_feedback_eligible[dynamics_feedback_gain]) if dynamics_feedback_eligible.repair.nunique() == 2 else np.nan
    rho, rho_p = spearmanr(dynamics_feedback_eligible[dynamics_feedback_gain], dynamics_feedback_eligible.repair)
    sample_effect = dynamics_feedback_eligible.groupby('sample_id').apply(lambda group: group.loc[group[dynamics_feedback_gain].idxmax(), 'repair'] - group.repair.mean(), include_groups=False).to_numpy(dtype=float)
    dynamics_feedback_estimates = np.asarray([bootstrap_rng.choice(sample_effect, len(sample_effect), replace=True).mean() for _ in range(dynamics_feedback_config['bootstrap_repetitions'])])
    model_auroc, dynamics_feedback_ = grouped_binary_score(dynamics_feedback_eligible, [dynamics_feedback_gain], 'repair')
    raw_gain_rows.append({
        'metric': dynamics_feedback_metric,
        'candidates': len(dynamics_feedback_eligible),
        'samples': dynamics_feedback_eligible.sample_id.nunique(),
        'repair_AUROC': auroc,
        'repair_spearman': rho,
        'repair_spearman_p_value': rho_p,
        'group_cross_validated_AUROC': model_auroc,
        'best_minus_mean_repair_effect': sample_effect.mean(),
        'bootstrap_ci_low': np.quantile(dynamics_feedback_estimates, 0.025),
        'bootstrap_ci_high': np.quantile(dynamics_feedback_estimates, 0.975),
    })

raw_gain_vs_repair = pd.DataFrame(raw_gain_rows)
raw_gain_vs_repair.to_csv(dynamics_feedback_data_paths.output('raw_gain_vs_repair_csv'), index=False)
raw_gain_vs_repair


## Experiments B–C — Metric ablation and overlap/gain joint analysis

Compare instability metrics and combine response with spatial overlap. The question is whether gain contributes predictive information beyond the location and size of the action.


In [ ]:
metric_rows = []
for dynamics_feedback_metric in RAW_METRICS + ('rank',):
    dynamics_feedback_gain = 'rank_gain' if dynamics_feedback_metric == 'rank' else f'{dynamics_feedback_metric}_gain'
    for task, dynamics_feedback_subset, dynamics_feedback_target in (('repair', raw_primary[raw_primary.failure_type != 'not_failure'], 'repair'), ('regression', raw_primary[raw_primary.failure_type == 'not_failure'], 'regression'), ('syndrome_progress', raw_primary, 'syndrome_progress_gain')):
        if task == 'syndrome_progress':
            binary = dynamics_feedback_subset.assign(syndrome_progress_positive=(dynamics_feedback_subset[dynamics_feedback_target] > 0).astype(int))
            target_name = 'syndrome_progress_positive'
        else:
            binary = dynamics_feedback_subset
            target_name = dynamics_feedback_target

        dynamics_feedback_score, dynamics_feedback_ = grouped_binary_score(binary, [dynamics_feedback_gain], target_name)
        effect = binary.groupby(target_name)[dynamics_feedback_gain].mean().to_dict()
        metric_rows.append({
            'metric': dynamics_feedback_metric,
            'task': task,
            'AUROC': dynamics_feedback_score,
            'positive_minus_negative_mean': effect.get(1, np.nan) - effect.get(0, np.nan),
        })

repair_frame = raw_primary[raw_primary.failure_type != 'not_failure'].copy()
models = {
    'residual_only': ['raw_trajectory_gain'],
    'conflict_only': ['extrinsic_conflict_gain'],
    'residual_plus_conflict': ['raw_trajectory_gain', 'extrinsic_conflict_gain'],
    'all_dynamics': [f'{name}_gain' for name in RAW_METRICS],
}

for dynamics_feedback_name, dynamics_feedback_features in models.items():
    dynamics_feedback_score, dynamics_feedback_ = grouped_binary_score(repair_frame, dynamics_feedback_features, 'repair')
    metric_rows.append({
        'metric': dynamics_feedback_name,
        'task': 'repair_multivariate',
        'AUROC': dynamics_feedback_score,
        'positive_minus_negative_mean': np.nan,
    })

instability_metric_ablation = pd.DataFrame(metric_rows)
instability_metric_ablation.to_csv(dynamics_feedback_data_paths.output('instability_metric_ablation_csv'), index=False)
joint_features = ['weighted_instability_overlap', 'raw_trajectory_gain', 'row_weight']
joint_auroc, joint_probability = grouped_binary_score(repair_frame, joint_features, 'repair')
repair_frame['joint_cv_probability'] = joint_probability
repair_frame['overlap_bin'] = pd.qcut(repair_frame.weighted_instability_overlap.rank(method='first'), 5, labels=False)
repair_frame['gain_bin'] = pd.qcut(repair_frame.raw_trajectory_gain.rank(method='first'), 5, labels=False)
overlap_gain_joint = repair_frame.groupby(['overlap_bin', 'gain_bin'], as_index=False).agg(
    candidates=('candidate_id', 'size'),
    samples=('sample_id', 'nunique'),
    repair_probability=('repair', 'mean'),
    mean_overlap=('weighted_instability_overlap', 'mean'),
    mean_raw_gain=('raw_trajectory_gain', 'mean'),
)
overlap_gain_joint['joint_group_cv_AUROC'] = joint_auroc

overlap_gain_joint.to_csv(dynamics_feedback_data_paths.output('overlap_gain_joint_analysis_csv'), index=False)
instability_metric_ablation


## Experiments D–E — Hybrid targeting and unbiased candidate-pool validation

Test hybrid targeting, then repeat the candidate comparison with the unbiased pool construction. This checks whether apparent selection quality depends on which candidates were offered.


In [ ]:
hybrid_scored = []
for dynamics_feedback_sample_id, dynamics_feedback_group in raw_primary.groupby('sample_id', sort=True):
    dynamics_feedback_group = dynamics_feedback_group.copy()
    dynamics_feedback_group['normalized_gain'] = minmax_within_group(dynamics_feedback_group.raw_trajectory_gain)
    dynamics_feedback_group['normalized_overlap'] = minmax_within_group(dynamics_feedback_group.weighted_instability_overlap)

    for alpha in NEXT_CONFIG['hybrid_alphas']:
        scored = dynamics_feedback_group.assign(hybrid_score=alpha * dynamics_feedback_group.normalized_gain + (1 - alpha) * dynamics_feedback_group.normalized_overlap)
        dynamics_feedback_selected = scored.sort_values(['hybrid_score', 'candidate_id'], ascending=[False, True]).iloc[0]
        hybrid_scored.append({
            'sample_id': dynamics_feedback_sample_id,
            'split': dynamics_feedback_selected.split,
            'failure_type': dynamics_feedback_selected.failure_type,
            'alpha': alpha,
            'candidate_id': dynamics_feedback_selected.candidate_id,
            'repair': dynamics_feedback_selected.repair,
            'regression': dynamics_feedback_selected.regression,
            'syndrome_consistent': dynamics_feedback_selected.syndrome_consistent,
            'logical_correct': dynamics_feedback_selected.logical_correct,
            'selection_edge_updates': int(dynamics_feedback_group.localization_edge_updates.iloc[0] + dynamics_feedback_group.probe_edge_updates.sum()),
            'decode_edge_updates': dynamics_feedback_selected.full_edge_updates,
            'total_edge_updates': int(dynamics_feedback_group.localization_edge_updates.iloc[0] + dynamics_feedback_group.probe_edge_updates.sum() + dynamics_feedback_selected.full_edge_updates),
            'selection_runtime_seconds': dynamics_feedback_group.localization_runtime_seconds.iloc[0] + dynamics_feedback_group.probe_runtime_seconds.sum(),
            'decode_runtime_seconds': dynamics_feedback_selected.full_runtime_seconds,
            'total_runtime_seconds': dynamics_feedback_group.localization_runtime_seconds.iloc[0] + dynamics_feedback_group.probe_runtime_seconds.sum() + dynamics_feedback_selected.full_runtime_seconds,
        })

hybrid_cases = pd.DataFrame(hybrid_scored)
development_hybrid = hybrid_cases[hybrid_cases.split == 'development']
alpha_scores = development_hybrid.groupby('alpha').agg(repair=('repair', 'sum'), regression=('regression', 'sum')).assign(net=lambda frame: frame.repair - frame.regression)
selected_alpha = float(alpha_scores.sort_values(['net', 'repair'], ascending=False).index[0])
target_rows = []

for dynamics_feedback_sample_id, dynamics_feedback_group in raw_primary[raw_primary.split == 'test'].groupby('sample_id', sort=True):
    dynamics_feedback_baseline_success = dynamics_feedback_group.failure_type.iloc[0] == 'not_failure'
    selections = {
        'anti_targeted': dynamics_feedback_group.sort_values(['weighted_instability_overlap', 'candidate_id']).iloc[0],
        'overlap_targeted': dynamics_feedback_group.sort_values(['weighted_instability_overlap', 'candidate_id'], ascending=[False, True]).iloc[0],
        'gain_targeted': dynamics_feedback_group.sort_values(['raw_trajectory_gain', 'candidate_id'], ascending=[False, True]).iloc[0],
    }

    dynamics_feedback_hybrid = hybrid_cases[(hybrid_cases.sample_id == dynamics_feedback_sample_id) & (hybrid_cases.alpha == selected_alpha)].iloc[0]
    selections['hybrid_targeted'] = dynamics_feedback_group[dynamics_feedback_group.candidate_id == dynamics_feedback_hybrid.candidate_id].iloc[0]
    dynamics_feedback_gain_choice = selections['gain_targeted']
    matched = dynamics_feedback_group[dynamics_feedback_group.row_weight == dynamics_feedback_gain_choice.row_weight]

    for dynamics_feedback_seed_index in range(NEXT_CONFIG['random_control_seeds']):
        dynamics_feedback_rng = np.random.default_rng(NEXT_CONFIG['next_random_seed'] + int(dynamics_feedback_sample_id) * 101 + dynamics_feedback_seed_index)
        selections[f'fully_random_{dynamics_feedback_seed_index}'] = dynamics_feedback_group.iloc[int(dynamics_feedback_rng.integers(len(dynamics_feedback_group)))]
        selections[f'weight_matched_random_{dynamics_feedback_seed_index}'] = matched.iloc[int(dynamics_feedback_rng.integers(len(matched)))]

    for label, dynamics_feedback_selected in selections.items():
        dynamics_feedback_strategy = label.rsplit('_', 1)[0] if label.rsplit('_', 1)[-1].isdigit() else label
        dynamics_feedback_success = bool(dynamics_feedback_selected.syndrome_consistent and dynamics_feedback_selected.logical_correct)
        target_rows.append({
            'sample_id': dynamics_feedback_sample_id,
            'failure_type': dynamics_feedback_selected.failure_type,
            'strategy': dynamics_feedback_strategy,
            'candidate_id': dynamics_feedback_selected.candidate_id,
            'repair': int(not dynamics_feedback_baseline_success and dynamics_feedback_success),
            'regression': int(dynamics_feedback_baseline_success and (not dynamics_feedback_success)),
            'syndrome_consistent': dynamics_feedback_selected.syndrome_consistent,
            'logical_correct': dynamics_feedback_selected.logical_correct,
            'selection_edge_updates': int(dynamics_feedback_group.localization_edge_updates.iloc[0] + dynamics_feedback_group.probe_edge_updates.sum()),
            'decode_edge_updates': dynamics_feedback_selected.full_edge_updates,
            'total_edge_updates': int(dynamics_feedback_group.localization_edge_updates.iloc[0] + dynamics_feedback_group.probe_edge_updates.sum() + dynamics_feedback_selected.full_edge_updates),
            'selection_runtime_seconds': dynamics_feedback_group.localization_runtime_seconds.iloc[0] + dynamics_feedback_group.probe_runtime_seconds.sum(),
            'decode_runtime_seconds': dynamics_feedback_selected.full_runtime_seconds,
            'total_runtime_seconds': dynamics_feedback_group.localization_runtime_seconds.iloc[0] + dynamics_feedback_group.probe_runtime_seconds.sum() + dynamics_feedback_selected.full_runtime_seconds,
        })

hybrid_targeting_cases = pd.DataFrame(target_rows)
hybrid_targeting = dynamics_feedback_conditional_outcome_rates(hybrid_targeting_cases.groupby('strategy', as_index=False).agg(
    evaluations=('sample_id', 'size'),
    unique_cases=('sample_id', 'nunique'),
    repair_count=('repair', 'sum'),
    regression_count=('regression', 'sum'),
    baseline_failure_evaluations=('failure_type', lambda x: np.sum(x != 'not_failure')),
    baseline_success_evaluations=('failure_type', lambda x: np.sum(x == 'not_failure')),
    syndrome_consistent_rate=('syndrome_consistent', 'mean'),
    mean_total_edge_updates=('total_edge_updates', 'mean'),
    median_total_runtime_seconds=('total_runtime_seconds', 'median'),
))
hybrid_targeting['selected_development_alpha'] = selected_alpha
hybrid_targeting['net_gain'] = hybrid_targeting_cases.groupby('strategy').apply(lambda group: group.repair.sum() - group.regression.sum(), include_groups=False).to_numpy()

hybrid_targeting.to_csv(dynamics_feedback_data_paths.output('hybrid_targeting_ablation_csv'), index=False)
pool_rows = []
for dynamics_feedback_sample_id, dynamics_feedback_group in raw_primary.groupby('sample_id', sort=True):
    overlap_order = dynamics_feedback_group.sort_values(['weighted_instability_overlap', 'candidate_id'], ascending=[False, True])
    uniform = dynamics_feedback_group.sample(n=min(len(dynamics_feedback_group), NEXT_CONFIG['unbiased_candidate_limit']), random_state=NEXT_CONFIG['next_random_seed'] + int(dynamics_feedback_sample_id))
    for pool_name, dynamics_feedback_pool in [('overlap_screened', overlap_order.head(NEXT_CONFIG['candidate_screen_limit'])), ('uniform_pool', uniform)]:
        pool_rows.append({
            'sample_id': dynamics_feedback_sample_id,
            'failure_type': dynamics_feedback_group.failure_type.iloc[0],
            'pool': pool_name,
            'candidates': len(dynamics_feedback_pool),
            'gain_repair_AUROC': roc_auc_score(dynamics_feedback_pool.repair, dynamics_feedback_pool.raw_trajectory_gain) if dynamics_feedback_pool.repair.nunique() == 2 else np.nan,
            'gain_repair_spearman': spearmanr(dynamics_feedback_pool.raw_trajectory_gain, dynamics_feedback_pool.repair).statistic,
        })

unbiased_candidate_validation = pd.DataFrame(pool_rows).groupby('pool', as_index=False).agg(
    samples=('sample_id', 'nunique'),
    mean_candidates=('candidates', 'mean'),
    mean_gain_repair_AUROC=('gain_repair_AUROC', 'mean'),
    mean_gain_repair_spearman=('gain_repair_spearman', 'mean'),
)
unbiased_candidate_validation.to_csv(dynamics_feedback_data_paths.output('unbiased_candidate_validation_csv'), index=False)
hybrid_targeting


## Experiment F — Stabilization versus useful trajectory redirection

Separate numerical stabilization from a useful trajectory change. A residual decrease with no correct-sector recovery is not a successful repair.


In [ ]:
redirection_rows = []
for dynamics_feedback_sample in tqdm(next_cases.itertuples(index=False), total=len(next_cases), desc='Experiment F: redirection', disable=NEXT_CONFIG['print_level'] == 1):
    dynamics_feedback_, dynamics_feedback_context = dynamics_feedback_context_for(2, dynamics_feedback_sample.p)
    dynamics_feedback_syndrome = dynamics_feedback_bit_array(dynamics_feedback_sample.syndrome)
    dynamics_feedback_base_trace = dynamics_feedback_trace_min_sum(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_config['benchmark_iterations'])
    base_metrics = raw_dynamics(dynamics_feedback_context, dynamics_feedback_base_trace, NEXT_CONFIG['probe_iterations'])
    dynamics_feedback_region = dynamics_feedback_top_region(base_metrics.rank_trajectory, min(NEXT_CONFIG['primary_top_k'], dynamics_feedback_context.H.shape[1]))
    dynamics_feedback_group = raw_primary[raw_primary.sample_id == dynamics_feedback_sample.sample_id]
    selected_rows = {
        'random_redundant_row': dynamics_feedback_group.sample(1, random_state=NEXT_CONFIG['next_random_seed']).iloc[0],
        'overlap_targeted_row': dynamics_feedback_group.sort_values('weighted_instability_overlap', ascending=False).iloc[0],
        'gain_targeted_row': dynamics_feedback_group.sort_values('raw_trajectory_gain', ascending=False).iloc[0],
    }

    for dynamics_feedback_name, dynamics_feedback_selected in selected_rows.items():
        redirection_rows.append({
            'sample_id': dynamics_feedback_sample.sample_id,
            'failure_type': dynamics_feedback_sample.failure_type,
            'intervention': dynamics_feedback_name,
            'raw_instability_reduction': dynamics_feedback_selected.raw_trajectory_gain,
            'syndrome_progress_gain': dynamics_feedback_selected.syndrome_progress_gain,
            'minimum_syndrome_weight': dynamics_feedback_selected.minimum_syndrome_weight_after,
            'mean_hard_movement': dynamics_feedback_selected.mean_hard_movement_after,
            'repair': dynamics_feedback_selected.repair,
            'regression': dynamics_feedback_selected.regression,
        })

    for dynamics_feedback_name, dynamics_feedback_damping in (('global_damping', np.full(len(dynamics_feedback_context.edge_checks), NEXT_CONFIG['damping_gamma_max'])), ('local_targeted_damping', dynamics_feedback_edge_maps(dynamics_feedback_context, base_metrics.raw_trajectory, NEXT_CONFIG['damping_gamma_min'], NEXT_CONFIG['damping_gamma_max'], 'targeted'))):
        dynamics_feedback_trace = dynamics_feedback_trace_min_sum(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_config['benchmark_iterations'], edge_damping=dynamics_feedback_damping)
        after = raw_dynamics(dynamics_feedback_context, dynamics_feedback_trace, NEXT_CONFIG['probe_iterations'])
        weights = syndrome_residual_weights(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_trace)
        dynamics_feedback_, dynamics_feedback_valid, dynamics_feedback_correct, dynamics_feedback_ = outcome_at_first_valid(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_sample.true_sector, dynamics_feedback_trace)
        dynamics_feedback_success = dynamics_feedback_valid and dynamics_feedback_correct
        redirection_rows.append({
            'sample_id': dynamics_feedback_sample.sample_id,
            'failure_type': dynamics_feedback_sample.failure_type,
            'intervention': dynamics_feedback_name,
            'raw_instability_reduction': metric_region_mean(base_metrics, 'raw_trajectory', dynamics_feedback_region) - metric_region_mean(after, 'raw_trajectory', dynamics_feedback_region),
            'syndrome_progress_gain': int(syndrome_residual_weights(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_base_trace)[NEXT_CONFIG['primary_candidate_test_iterations']] - weights[NEXT_CONFIG['primary_candidate_test_iterations']]),
            'minimum_syndrome_weight': int(weights.min()),
            'mean_hard_movement': float(hard_decision_movement(dynamics_feedback_trace).mean()),
            'repair': int(dynamics_feedback_sample.failure_type != 'not_failure' and dynamics_feedback_success),
            'regression': int(dynamics_feedback_sample.failure_type == 'not_failure' and (not dynamics_feedback_success)),
        })

trajectory_redirection = pd.DataFrame(redirection_rows).groupby('intervention', as_index=False).agg(
    cases=('sample_id', 'size'),
    mean_raw_instability_reduction=('raw_instability_reduction', 'mean'),
    mean_syndrome_progress_gain=('syndrome_progress_gain', 'mean'),
    mean_minimum_syndrome_weight=('minimum_syndrome_weight', 'mean'),
    mean_hard_movement=('mean_hard_movement', 'mean'),
    repair_rate=('repair', 'mean'),
    regression_rate=('regression', 'mean'),
)
trajectory_redirection.to_csv(dynamics_feedback_data_paths.output('trajectory_redirection_analysis_csv'), index=False)
trajectory_redirection


## Experiment G — Early intervention for stable logical-wrong cases

Intervene earlier on stable logical-wrong trajectories to test whether late instability-based triggers miss this failure family. Ground-truth classification remains an offline analysis label.


In [ ]:
early_input = next_cases[next_cases.failure_type.isin(['stable_persistent_logical_failure', 'not_failure'])].copy()
early_rows = []
for dynamics_feedback_sample in tqdm(early_input.itertuples(index=False), total=len(early_input), desc='Experiment G: early intervention', disable=NEXT_CONFIG['print_level'] == 1):
    dynamics_feedback_, dynamics_feedback_context = dynamics_feedback_context_for(2, dynamics_feedback_sample.p)
    dynamics_feedback_syndrome = dynamics_feedback_bit_array(dynamics_feedback_sample.syndrome)
    dynamics_feedback_candidates = dynamics_feedback_generate_redundant_candidates(dynamics_feedback_context.H, NEXT_CONFIG['candidate_max_row_weight'])

    for probe_time in NEXT_CONFIG['early_probe_iterations']:
        localization_started = perf_counter()
        localization_iterations = max(probe_time, NEXT_CONFIG['primary_candidate_test_iterations'])
        dynamics_feedback_baseline_trace = dynamics_feedback_trace_min_sum(dynamics_feedback_context, dynamics_feedback_syndrome, localization_iterations)
        localization_runtime = perf_counter() - localization_started
        localization_edge_updates = localization_iterations * int(dynamics_feedback_context.H.sum())
        early_metrics = raw_dynamics(dynamics_feedback_context, dynamics_feedback_baseline_trace, probe_time)
        dynamics_feedback_region = dynamics_feedback_top_region(early_metrics.rank_trajectory, min(NEXT_CONFIG['primary_top_k'], dynamics_feedback_context.H.shape[1]))
        overlap = np.asarray([dynamics_feedback_candidate_overlap(candidate, early_metrics.rank_trajectory, dynamics_feedback_region)[1] for candidate in dynamics_feedback_candidates])
        dynamics_feedback_screened = balanced_candidate_screen(dynamics_feedback_candidates, overlap, NEXT_CONFIG['candidate_screen_limit'], NEXT_CONFIG['next_random_seed'] + int(dynamics_feedback_sample.sample_id) * 4099 + probe_time)
        dynamics_feedback_before = {value: raw_dynamics(dynamics_feedback_context, dynamics_feedback_baseline_trace, value) for value in NEXT_CONFIG['candidate_test_iterations']}
        evaluated = []

        for dynamics_feedback_candidate in dynamics_feedback_screened:
            dynamics_feedback_probes = probe_raw_candidate(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_candidate, dynamics_feedback_region, dynamics_feedback_before, dynamics_feedback_baseline_trace)
            dynamics_feedback_primary = [row for row in dynamics_feedback_probes if row['test_iterations'] == NEXT_CONFIG['primary_candidate_test_iterations']][0]
            evaluated.append((dynamics_feedback_candidate, dynamics_feedback_primary))

        gain_selected = max(evaluated, key=lambda item: (item[1]['raw_trajectory_gain'], -item[0].candidate_id))
        anti_selected = min(evaluated, key=lambda item: (overlap[item[0].candidate_id], item[0].candidate_id))
        selections = [('instability_targeted', gain_selected), ('anti_targeted', anti_selected)]

        for dynamics_feedback_seed_index in range(NEXT_CONFIG['random_control_seeds']):
            dynamics_feedback_rng = np.random.default_rng(NEXT_CONFIG['next_random_seed'] + int(dynamics_feedback_sample.sample_id) * 1009 + probe_time * 17 + dynamics_feedback_seed_index)
            selections.append(('random', evaluated[int(dynamics_feedback_rng.integers(len(evaluated)))]))

        full_cache = {}
        for dynamics_feedback_strategy, (dynamics_feedback_candidate, dynamics_feedback_primary) in selections:
            if dynamics_feedback_candidate.candidate_id not in full_cache:
                full_cache[dynamics_feedback_candidate.candidate_id] = full_raw_candidate(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_sample.true_sector, dynamics_feedback_candidate, dynamics_feedback_region)
            dynamics_feedback_full = full_cache[dynamics_feedback_candidate.candidate_id]
            dynamics_feedback_success = bool(dynamics_feedback_full['syndrome_consistent'] and dynamics_feedback_full['logical_correct'])
            early_rows.append({
                'sample_id': dynamics_feedback_sample.sample_id,
                'failure_type': dynamics_feedback_sample.failure_type,
                'probe_iterations': probe_time,
                'strategy': dynamics_feedback_strategy,
                'candidate_id': dynamics_feedback_candidate.candidate_id,
                'raw_trajectory_gain': dynamics_feedback_primary['raw_trajectory_gain'],
                'syndrome_consistent': dynamics_feedback_full['syndrome_consistent'],
                'logical_correct': dynamics_feedback_full['logical_correct'],
                'logical_repair': int(dynamics_feedback_sample.failure_type == 'stable_persistent_logical_failure' and dynamics_feedback_success),
                'regression': int(dynamics_feedback_sample.failure_type == 'not_failure' and (not dynamics_feedback_success)),
                'selection_edge_updates': localization_edge_updates + sum((item[1]['probe_edge_updates'] for item in evaluated)),
                'decode_edge_updates': dynamics_feedback_full['full_edge_updates'],
                'total_edge_updates': localization_edge_updates + sum((item[1]['probe_edge_updates'] for item in evaluated)) + dynamics_feedback_full['full_edge_updates'],
                'selection_runtime_seconds': localization_runtime + sum((item[1]['probe_runtime_seconds'] for item in evaluated)),
                'decode_runtime_seconds': dynamics_feedback_full['full_runtime_seconds'],
                'total_runtime_seconds': localization_runtime + sum((item[1]['probe_runtime_seconds'] for item in evaluated)) + dynamics_feedback_full['full_runtime_seconds'],
            })

early_intervention_trials = pd.DataFrame(early_rows)
early_intervention_trials.to_csv(dynamics_feedback_data_paths.output('early_intervention_trials_csv'), index=False)
early_intervention_summary = dynamics_feedback_conditional_outcome_rates(early_intervention_trials.groupby(['probe_iterations', 'strategy'], as_index=False).agg(
    evaluations=('sample_id', 'size'),
    unique_cases=('sample_id', 'nunique'),
    repair_count=('logical_repair', 'sum'),
    regression_count=('regression', 'sum'),
    baseline_failure_evaluations=('failure_type', lambda values: np.sum(values == 'stable_persistent_logical_failure')),
    baseline_success_evaluations=('failure_type', lambda values: np.sum(values == 'not_failure')),
    mean_raw_gain=('raw_trajectory_gain', 'mean'),
    mean_total_edge_updates=('total_edge_updates', 'mean'),
    median_total_runtime_seconds=('total_runtime_seconds', 'median'),
))

early_intervention_summary.to_csv(dynamics_feedback_data_paths.output('early_intervention_stable_wrong_csv'), index=False)
early_intervention_summary


## Experiments H–I — Fixed-region iterative feedback and stopping rules

Test repeated feedback using the fixed region and compare stopping decisions. Keeping the region fixed makes successive response measurements comparable.


In [ ]:
feedback_rows = []
feedback_final_rows = []
for dynamics_feedback_sample in tqdm(next_cases.itertuples(index=False), total=len(next_cases), desc='Experiments H-I: feedback', disable=NEXT_CONFIG['print_level'] == 1):
    dynamics_feedback_, dynamics_feedback_base_context = dynamics_feedback_context_for(2, dynamics_feedback_sample.p)
    dynamics_feedback_base_syndrome = dynamics_feedback_bit_array(dynamics_feedback_sample.syndrome)
    dynamics_feedback_current = dynamics_feedback_base_context
    dynamics_feedback_current_syndrome = dynamics_feedback_base_syndrome.copy()
    initial_trace = dynamics_feedback_trace_min_sum(dynamics_feedback_current, dynamics_feedback_current_syndrome, NEXT_CONFIG['probe_iterations'])
    initial_metrics = raw_dynamics(dynamics_feedback_current, initial_trace, NEXT_CONFIG['probe_iterations'])
    fixed_region = dynamics_feedback_top_region(initial_metrics.rank_trajectory, min(NEXT_CONFIG['primary_top_k'], dynamics_feedback_current.H.shape[1]))
    dynamics_feedback_cumulative_updates = 0
    cumulative_runtime = 0.0

    for dynamics_feedback_round_index in range(NEXT_CONFIG['iterative_max_rounds'] + 1):
        started = perf_counter()
        dynamics_feedback_probe = dynamics_feedback_trace_min_sum(dynamics_feedback_current, dynamics_feedback_current_syndrome, NEXT_CONFIG['probe_iterations'])
        probe_runtime = perf_counter() - started
        probe_edge_updates = NEXT_CONFIG['probe_iterations'] * int(dynamics_feedback_current.H.sum())
        dynamics_feedback_cumulative_updates += probe_edge_updates
        cumulative_runtime += probe_runtime
        metrics = raw_dynamics(dynamics_feedback_current, dynamics_feedback_probe, NEXT_CONFIG['probe_iterations'])
        current_region = dynamics_feedback_top_region(metrics.rank_trajectory, min(NEXT_CONFIG['primary_top_k'], dynamics_feedback_current.H.shape[1]))
        fixed_I = metric_region_mean(metrics, 'raw_trajectory', fixed_region)
        dynamics_feedback_current_I = metric_region_mean(metrics, 'raw_trajectory', current_region)
        global_I = float(metrics.raw_trajectory.mean())
        residual_weights = syndrome_residual_weights(dynamics_feedback_current, dynamics_feedback_current_syndrome, dynamics_feedback_probe)
        dynamics_feedback_, dynamics_feedback_valid, dynamics_feedback_correct, dynamics_feedback_ = outcome_at_first_valid(dynamics_feedback_base_context, dynamics_feedback_base_syndrome, dynamics_feedback_sample.true_sector, dynamics_feedback_probe)
        dynamics_feedback_row = {
            'sample_id': dynamics_feedback_sample.sample_id,
            'split': dynamics_feedback_sample.split,
            'failure_type': dynamics_feedback_sample.failure_type,
            'round': dynamics_feedback_round_index,
            'fixed_region_instability': fixed_I,
            'current_region_instability': dynamics_feedback_current_I,
            'global_instability': global_I,
            'syndrome_residual_weight': int(residual_weights[-1]),
            'syndrome_consistent': int(dynamics_feedback_valid),
            'logical_correct': int(dynamics_feedback_correct),
            'selected_candidate_id': np.nan,
            'raw_gain': np.nan,
            'overlap_score': np.nan,
            'round_edge_updates': probe_edge_updates,
            'cumulative_edge_updates': dynamics_feedback_cumulative_updates,
            'round_runtime_seconds': probe_runtime,
            'cumulative_runtime_seconds': cumulative_runtime,
            'no_positive_gain': 0,
        }

        feedback_rows.append(dynamics_feedback_row)
        if dynamics_feedback_round_index == NEXT_CONFIG['iterative_max_rounds'] or dynamics_feedback_valid:
            break
        dynamics_feedback_candidates = dynamics_feedback_generate_redundant_candidates(dynamics_feedback_current.H, NEXT_CONFIG['candidate_max_row_weight'])
        overlaps = np.asarray([dynamics_feedback_candidate_overlap(candidate, metrics.rank_trajectory, current_region)[1] for candidate in dynamics_feedback_candidates])
        order = np.lexsort((np.arange(len(dynamics_feedback_candidates)), -overlaps))[:NEXT_CONFIG['candidate_screen_limit']]
        dynamics_feedback_screened = [dynamics_feedback_candidates[index] for index in order]
        dynamics_feedback_before = {value: raw_dynamics(dynamics_feedback_current, dynamics_feedback_probe, value) for value in NEXT_CONFIG['candidate_test_iterations']}
        dynamics_feedback_measured = []

        for dynamics_feedback_candidate in dynamics_feedback_screened:
            dynamics_feedback_augmented = dynamics_feedback_augmented_context(dynamics_feedback_current, dynamics_feedback_candidate.row)
            dynamics_feedback_augmented_s = dynamics_feedback_augmented_syndrome(dynamics_feedback_current_syndrome, dynamics_feedback_candidate.source_rows)
            started = perf_counter()
            test_trace = dynamics_feedback_trace_min_sum(dynamics_feedback_augmented, dynamics_feedback_augmented_s, NEXT_CONFIG['primary_candidate_test_iterations'])
            elapsed = perf_counter() - started
            after = raw_dynamics(dynamics_feedback_augmented, test_trace, NEXT_CONFIG['primary_candidate_test_iterations'])
            dynamics_feedback_gain = metric_region_mean(dynamics_feedback_before[NEXT_CONFIG['primary_candidate_test_iterations']], 'raw_trajectory', current_region) - metric_region_mean(after, 'raw_trajectory', current_region)
            updates = NEXT_CONFIG['primary_candidate_test_iterations'] * int(dynamics_feedback_augmented.H.sum())
            dynamics_feedback_measured.append((dynamics_feedback_gain, dynamics_feedback_candidate, elapsed, updates))
            cumulative_runtime += elapsed
            dynamics_feedback_cumulative_updates += updates

        dynamics_feedback_gain, dynamics_feedback_selected, dynamics_feedback_, dynamics_feedback_ = max(dynamics_feedback_measured, key=lambda item: (item[0], -item[1].candidate_id))
        feedback_rows[-1].update({
            'selected_candidate_id': dynamics_feedback_selected.candidate_id,
            'raw_gain': dynamics_feedback_gain,
            'overlap_score': dynamics_feedback_candidate_overlap(dynamics_feedback_selected, metrics.rank_trajectory, current_region)[1],
            'round_edge_updates': probe_edge_updates + sum((item[3] for item in dynamics_feedback_measured)),
            'cumulative_edge_updates': dynamics_feedback_cumulative_updates,
            'cumulative_runtime_seconds': cumulative_runtime,
            'no_positive_gain': int(dynamics_feedback_gain <= 0),
        })

        if dynamics_feedback_gain <= 0:
            break
        dynamics_feedback_current_syndrome = dynamics_feedback_augmented_syndrome(dynamics_feedback_current_syndrome, dynamics_feedback_selected.source_rows)
        dynamics_feedback_current = dynamics_feedback_augmented_context(dynamics_feedback_current, dynamics_feedback_selected.row)

    sample_rows = [row for row in feedback_rows if row['sample_id'] == dynamics_feedback_sample.sample_id]
    dynamics_feedback_last = sample_rows[-1]
    started = perf_counter()
    final_trace = dynamics_feedback_trace_min_sum(dynamics_feedback_current, dynamics_feedback_current_syndrome, dynamics_feedback_config['benchmark_iterations'])
    final_runtime = perf_counter() - started
    final_index, final_valid, final_correct, dynamics_feedback_ = outcome_at_first_valid(dynamics_feedback_base_context, dynamics_feedback_base_syndrome, dynamics_feedback_sample.true_sector, final_trace)
    dynamics_feedback_cumulative_updates += int(dynamics_feedback_config['benchmark_iterations'] * dynamics_feedback_current.H.sum())
    cumulative_runtime += final_runtime
    dynamics_feedback_success = bool(final_valid and final_correct)
    feedback_final_rows.append({
        'sample_id': dynamics_feedback_sample.sample_id,
        'split': dynamics_feedback_sample.split,
        'failure_type': dynamics_feedback_sample.failure_type,
        'rounds': dynamics_feedback_last['round'],
        'initial_fixed_instability': sample_rows[0]['fixed_region_instability'],
        'final_fixed_instability': dynamics_feedback_last['fixed_region_instability'],
        'initial_global_instability': sample_rows[0]['global_instability'],
        'final_global_instability': dynamics_feedback_last['global_instability'],
        'syndrome_consistent': int(final_valid),
        'logical_correct': int(final_correct),
        'repair': int(dynamics_feedback_sample.failure_type != 'not_failure' and dynamics_feedback_success),
        'regression': int(dynamics_feedback_sample.failure_type == 'not_failure' and (not dynamics_feedback_success)),
        'edge_updates': dynamics_feedback_cumulative_updates,
        'runtime_seconds': cumulative_runtime,
    })

iterative_feedback_trajectories = pd.DataFrame(feedback_rows)
iterative_feedback_trajectories.to_csv(dynamics_feedback_data_paths.output('iterative_feedback_trajectories_csv'), index=False)
fixed_region_feedback = pd.DataFrame(feedback_final_rows)
fixed_region_feedback_summary = fixed_region_feedback.groupby('failure_type', as_index=False).agg(
    cases=('sample_id', 'size'),
    repair_rate=('repair', 'mean'),
    regression_rate=('regression', 'mean'),
    mean_rounds=('rounds', 'mean'),
    mean_fixed_reduction=('initial_fixed_instability', lambda values: np.nan),
    mean_global_reduction=('initial_global_instability', lambda values: np.nan),
    mean_edge_updates=('edge_updates', 'mean'),
    median_runtime_seconds=('runtime_seconds', 'median'),
)

for dynamics_feedback_index, summary_row in fixed_region_feedback_summary.iterrows():
    dynamics_feedback_subset = fixed_region_feedback[fixed_region_feedback.failure_type == summary_row.failure_type]
    fixed_region_feedback_summary.loc[dynamics_feedback_index, 'mean_fixed_reduction'] = (dynamics_feedback_subset.initial_fixed_instability - dynamics_feedback_subset.final_fixed_instability).mean()
    fixed_region_feedback_summary.loc[dynamics_feedback_index, 'mean_global_reduction'] = (dynamics_feedback_subset.initial_global_instability - dynamics_feedback_subset.final_global_instability).mean()

fixed_region_feedback_summary.to_csv(dynamics_feedback_data_paths.output('fixed_region_feedback_summary_csv'), index=False)
development_feedback = iterative_feedback_trajectories[iterative_feedback_trajectories['split'] == 'development']
global_thresholds = {q: development_feedback.global_instability.quantile(q) for q in NEXT_CONFIG['gating_quantiles']}
stopping_rows = []

for dynamics_feedback_sample_id, dynamics_feedback_group in iterative_feedback_trajectories[iterative_feedback_trajectories['split'] == 'test'].groupby('sample_id', sort=True):
    dynamics_feedback_group = dynamics_feedback_group.sort_values('round').reset_index(drop=True)
    rules = {
        'syndrome_consistent': dynamics_feedback_group.index[dynamics_feedback_group.syndrome_consistent.astype(bool)].min() if dynamics_feedback_group.syndrome_consistent.any() else len(dynamics_feedback_group) - 1,
        'no_positive_gain': dynamics_feedback_group.index[dynamics_feedback_group.no_positive_gain.astype(bool)].min() if dynamics_feedback_group.no_positive_gain.any() else len(dynamics_feedback_group) - 1,
    }

    for quantile, threshold in global_thresholds.items():
        match = dynamics_feedback_group.index[dynamics_feedback_group.syndrome_consistent.astype(bool) & (dynamics_feedback_group.global_instability < threshold)]
        rules[f'syndrome_consistent_low_global_q{quantile}'] = int(match.min()) if len(match) else len(dynamics_feedback_group) - 1

    for delta in NEXT_CONFIG['plateau_delta_values']:
        dynamics_feedback_reductions = dynamics_feedback_group.fixed_region_instability.shift(1) - dynamics_feedback_group.fixed_region_instability
        plateau = dynamics_feedback_reductions < delta
        stop = len(dynamics_feedback_group) - 1

        for dynamics_feedback_index in range(NEXT_CONFIG['plateau_patience'], len(dynamics_feedback_group)):
            if plateau.iloc[dynamics_feedback_index - NEXT_CONFIG['plateau_patience'] + 1:dynamics_feedback_index + 1].all():
                stop = dynamics_feedback_index
                break

        rules[f'plateau_{delta:g}'] = stop

    for rule, stop_index in rules.items():
        dynamics_feedback_selected = dynamics_feedback_group.iloc[int(stop_index)]
        dynamics_feedback_success = bool(dynamics_feedback_selected.syndrome_consistent and dynamics_feedback_selected.logical_correct)
        stopping_rows.append({
            'sample_id': dynamics_feedback_sample_id,
            'failure_type': dynamics_feedback_selected.failure_type,
            'rule': rule,
            'stop_round': dynamics_feedback_selected['round'],
            'repair': int(dynamics_feedback_selected.failure_type != 'not_failure' and dynamics_feedback_success),
            'regression': int(dynamics_feedback_selected.failure_type == 'not_failure' and (not dynamics_feedback_success)),
            'edge_updates': dynamics_feedback_selected.cumulative_edge_updates,
            'runtime_seconds': dynamics_feedback_selected.cumulative_runtime_seconds,
        })

feedback_stopping_cases = pd.DataFrame(stopping_rows)
feedback_stopping_cases.to_csv(dynamics_feedback_data_paths.output('feedback_stopping_cases_csv'), index=False)
feedback_stopping_rules = dynamics_feedback_conditional_outcome_rates(feedback_stopping_cases.groupby('rule', as_index=False).agg(
    cases=('sample_id', 'size'),
    repair_count=('repair', 'sum'),
    regression_count=('regression', 'sum'),
    baseline_failure_evaluations=('failure_type', lambda values: np.sum(values != 'not_failure')),
    baseline_success_evaluations=('failure_type', lambda values: np.sum(values == 'not_failure')),
    mean_stop_round=('stop_round', 'mean'),
    mean_edge_updates=('edge_updates', 'mean'),
    median_runtime_seconds=('runtime_seconds', 'median'),
))

feedback_stopping_rules.to_csv(dynamics_feedback_data_paths.output('feedback_stopping_rules_csv'), index=False)
fixed_region_feedback_summary


## Experiments J–K — Intervention gating and candidate-budget ablation

Vary the intervention gate and candidate budget. Track regressions and computational effort as well as failure-cohort repair so that additional search is not mistaken for a better signal.


In [ ]:
development_best = raw_primary[raw_primary.split == 'development'].sort_values(['sample_id', 'raw_trajectory_gain', 'candidate_id'], ascending=[True, False, True]).groupby('sample_id', as_index=False).head(1)
test_best = raw_primary[raw_primary.split == 'test'].sort_values(['sample_id', 'raw_trajectory_gain', 'candidate_id'], ascending=[True, False, True]).groupby('sample_id', as_index=False).head(1)
gain_thresholds = {q: development_best.raw_trajectory_gain.quantile(q) for q in NEXT_CONFIG['gating_quantiles']}
global_by_sample = iterative_feedback_trajectories[iterative_feedback_trajectories['round'] == 0].set_index('sample_id').global_instability
gating_rows = []

for quantile in NEXT_CONFIG['gating_quantiles']:
    gain_threshold = gain_thresholds[quantile]
    global_threshold = global_thresholds[quantile]
    for trigger in ('probe_invalid', 'global', 'gain', 'invalid_and_gain'):
        for dynamics_feedback_selected in test_best.itertuples(index=False):
            dynamics_feedback_baseline_success = dynamics_feedback_selected.failure_type == 'not_failure'
            if trigger == 'probe_invalid':
                intervene = not bool(dynamics_feedback_selected.short_probe_syndrome_consistent)
            elif trigger == 'global':
                intervene = global_by_sample.loc[dynamics_feedback_selected.sample_id] > global_threshold
            elif trigger == 'gain':
                intervene = dynamics_feedback_selected.raw_trajectory_gain > gain_threshold
            else:
                intervene = not bool(dynamics_feedback_selected.short_probe_syndrome_consistent) and dynamics_feedback_selected.raw_trajectory_gain > gain_threshold

            if intervene:
                dynamics_feedback_success = bool(dynamics_feedback_selected.syndrome_consistent and dynamics_feedback_selected.logical_correct)
            else:
                case = next_cases[next_cases.sample_id == dynamics_feedback_selected.sample_id].iloc[0]
                dynamics_feedback_success = bool(case.benchmark_syndrome_consistent and case.benchmark_logical_correct)

            gating_rows.append({
                'sample_id': dynamics_feedback_selected.sample_id,
                'failure_type': dynamics_feedback_selected.failure_type,
                'trigger': trigger,
                'quantile': quantile,
                'intervened': int(intervene),
                'repair': int(not dynamics_feedback_baseline_success and dynamics_feedback_success),
                'regression': int(dynamics_feedback_baseline_success and (not dynamics_feedback_success)),
                'net_gain': int(not dynamics_feedback_baseline_success and dynamics_feedback_success) - int(dynamics_feedback_baseline_success and (not dynamics_feedback_success)),
            })

intervention_gating = dynamics_feedback_conditional_outcome_rates(pd.DataFrame(gating_rows).groupby(['trigger', 'quantile'], as_index=False).agg(
    cases=('sample_id', 'size'),
    intervention_rate=('intervened', 'mean'),
    repair_count=('repair', 'sum'),
    regression_count=('regression', 'sum'),
    baseline_failure_evaluations=('failure_type', lambda values: np.sum(values != 'not_failure')),
    baseline_success_evaluations=('failure_type', lambda values: np.sum(values == 'not_failure')),
    net_gain=('net_gain', 'sum'),
))
intervention_gating.to_csv(dynamics_feedback_data_paths.output('intervention_gating_csv'), index=False)
budget_rows = []

for dynamics_feedback_sample_id, dynamics_feedback_group in raw_primary.groupby('sample_id', sort=True):
    overlap_order = dynamics_feedback_group.sort_values(['weighted_instability_overlap', 'candidate_id'], ascending=[False, True]).head(NEXT_CONFIG['candidate_screen_limit'])
    for budget in NEXT_CONFIG['candidate_budget_values']:
        dynamics_feedback_pool = overlap_order.head(budget)
        dynamics_feedback_selected = dynamics_feedback_pool.sort_values(['raw_trajectory_gain', 'candidate_id'], ascending=[False, True]).iloc[0]
        budget_rows.append({
            'sample_id': dynamics_feedback_sample_id,
            'split': dynamics_feedback_selected.split,
            'failure_type': dynamics_feedback_selected.failure_type,
            'search': 'fixed',
            'budget': budget,
            'probes_used': len(dynamics_feedback_pool),
            'candidate_id': dynamics_feedback_selected.candidate_id,
            'repair': dynamics_feedback_selected.repair,
            'regression': dynamics_feedback_selected.regression,
            'edge_updates': int(dynamics_feedback_group.localization_edge_updates.iloc[0] + dynamics_feedback_pool.probe_edge_updates.sum() + dynamics_feedback_selected.full_edge_updates),
            'runtime_seconds': dynamics_feedback_group.localization_runtime_seconds.iloc[0] + dynamics_feedback_pool.probe_runtime_seconds.sum() + dynamics_feedback_selected.full_runtime_seconds,
        })

    gains = overlap_order.raw_trajectory_gain.to_numpy()
    for quantile in NEXT_CONFIG['progressive_gain_quantiles']:
        threshold = development_best.raw_trajectory_gain.quantile(quantile)
        for patience in NEXT_CONFIG['progressive_patience_values']:
            best_index = 0
            best_gain = -np.inf
            stale = 0
            stop = len(overlap_order)

            for dynamics_feedback_position, dynamics_feedback_value in enumerate(gains):
                if dynamics_feedback_value > best_gain:
                    best_gain = dynamics_feedback_value
                    best_index = dynamics_feedback_position
                    stale = 0
                else:
                    stale += 1

                if best_gain > threshold or stale >= patience:
                    stop = dynamics_feedback_position + 1
                    break

            dynamics_feedback_pool = overlap_order.iloc[:stop]
            dynamics_feedback_selected = overlap_order.iloc[best_index]
            budget_rows.append({
                'sample_id': dynamics_feedback_sample_id,
                'split': dynamics_feedback_selected.split,
                'failure_type': dynamics_feedback_selected.failure_type,
                'search': f'progressive_q{quantile}_patience{patience}',
                'budget': NEXT_CONFIG['candidate_screen_limit'],
                'probes_used': len(dynamics_feedback_pool),
                'candidate_id': dynamics_feedback_selected.candidate_id,
                'repair': dynamics_feedback_selected.repair,
                'regression': dynamics_feedback_selected.regression,
                'edge_updates': int(dynamics_feedback_group.localization_edge_updates.iloc[0] + dynamics_feedback_pool.probe_edge_updates.sum() + dynamics_feedback_selected.full_edge_updates),
                'runtime_seconds': dynamics_feedback_group.localization_runtime_seconds.iloc[0] + dynamics_feedback_pool.probe_runtime_seconds.sum() + dynamics_feedback_selected.full_runtime_seconds,
            })

candidate_budget_trials = pd.DataFrame(budget_rows)
candidate_budget_trials.to_csv(dynamics_feedback_data_paths.output('candidate_budget_trials_csv'), index=False)
candidate_budget_ablation = dynamics_feedback_conditional_outcome_rates(candidate_budget_trials.groupby(['search', 'budget'], as_index=False).agg(
    cases=('sample_id', 'size'),
    mean_probes=('probes_used', 'mean'),
    repair_count=('repair', 'sum'),
    regression_count=('regression', 'sum'),
    baseline_failure_evaluations=('failure_type', lambda values: np.sum(values != 'not_failure')),
    baseline_success_evaluations=('failure_type', lambda values: np.sum(values == 'not_failure')),
    mean_edge_updates=('edge_updates', 'mean'),
    median_runtime_seconds=('runtime_seconds', 'median'),
))

candidate_budget_ablation.to_csv(dynamics_feedback_data_paths.output('candidate_budget_ablation_csv'), index=False)
joint_tuning_rows = []
development_primary = raw_primary[raw_primary['split'] == 'development']

for budget in NEXT_CONFIG['candidate_budget_values']:
    for alpha in NEXT_CONFIG['hybrid_alphas']:
        for quantile, threshold in gain_thresholds.items():
            outcomes = []
            for dynamics_feedback_sample_id, dynamics_feedback_group in development_primary.groupby('sample_id', sort=True):
                dynamics_feedback_pool = dynamics_feedback_group.sort_values(['weighted_instability_overlap', 'candidate_id'], ascending=[False, True]).head(budget)
                gains = minmax_within_group(dynamics_feedback_pool.raw_trajectory_gain.to_numpy())
                overlaps = minmax_within_group(dynamics_feedback_pool.weighted_instability_overlap.to_numpy())
                dynamics_feedback_selected = dynamics_feedback_pool.iloc[int(np.argmax(alpha * gains + (1 - alpha) * overlaps))]
                dynamics_feedback_baseline_success = dynamics_feedback_selected.failure_type == 'not_failure'
                intervene = dynamics_feedback_selected.raw_trajectory_gain > threshold
                dynamics_feedback_success = bool(dynamics_feedback_selected.syndrome_consistent and dynamics_feedback_selected.logical_correct) if intervene else dynamics_feedback_baseline_success
                outcomes.append({
                    'repair': int(not dynamics_feedback_baseline_success and dynamics_feedback_success),
                    'regression': int(dynamics_feedback_baseline_success and (not dynamics_feedback_success)),
                    'edge_updates': int(dynamics_feedback_group.localization_edge_updates.iloc[0] + dynamics_feedback_pool.probe_edge_updates.sum() + (dynamics_feedback_selected.full_edge_updates if intervene else 0)),
                })

            outcomes = pd.DataFrame(outcomes)
            joint_tuning_rows.append({
                'candidate_budget': budget,
                'hybrid_alpha': alpha,
                'gain_threshold_quantile': quantile,
                'gain_threshold': threshold,
                'repair': int(outcomes.repair.sum()),
                'regression': int(outcomes.regression.sum()),
                'net': int(outcomes.repair.sum() - outcomes.regression.sum()),
                'mean_edge_updates': outcomes.edge_updates.mean(),
            })

final_decoder_development_tuning = pd.DataFrame(joint_tuning_rows).sort_values(['net', 'repair', 'regression', 'mean_edge_updates', 'candidate_budget'], ascending=[False, False, True, True, True]).reset_index(drop=True)
final_decoder_development_tuning['selected'] = 0
final_decoder_development_tuning.loc[0, 'selected'] = 1
selected_tuning = final_decoder_development_tuning.iloc[0]
selected_candidate_budget = int(selected_tuning.candidate_budget)
selected_alpha = float(selected_tuning.hybrid_alpha)
selected_gain_threshold = float(selected_tuning.gain_threshold)

final_decoder_development_tuning.to_csv(dynamics_feedback_data_paths.output('final_decoder_development_tuning_csv'), index=False)
candidate_budget_ablation


## Experiments L–M — Candidate structure and failure-type mechanism

Inspect candidate structure and failure-family effects to identify where targeting is useful or harmful. These conditional results do not replace the full stochastic benchmark.


In [ ]:
structure_rows = []
for dynamics_feedback_row in raw_primary.itertuples(index=False):
    dynamics_feedback_, dynamics_feedback_context = dynamics_feedback_context_for(2, dynamics_feedback_row.p)
    dynamics_feedback_support = np.asarray(json.loads(dynamics_feedback_row.row_support), dtype=int)
    dynamics_feedback_source_rows = np.asarray(json.loads(dynamics_feedback_row.source_rows), dtype=int)

    touched_checks = np.flatnonzero(dynamics_feedback_context.H[:, dynamics_feedback_support].any(axis=1))
    subgraph = nx.Graph()
    subgraph.add_nodes_from([f'v{value}' for value in dynamics_feedback_support])
    subgraph.add_nodes_from([f'c{value}' for value in touched_checks])
    subgraph.add_edges_from(((f'c{check}', f'v{variable}') for check in touched_checks for variable in dynamics_feedback_support if dynamics_feedback_context.H[check, variable]))
    before_rank = len(subgraph.edges) - len(subgraph.nodes) + nx.number_connected_components(subgraph)
    augmented_graph = subgraph.copy()
    augmented_graph.add_node('new_check')
    augmented_graph.add_edges_from((('new_check', f'v{variable}') for variable in dynamics_feedback_support))
    after_rank = len(augmented_graph.edges) - len(augmented_graph.nodes) + nx.number_connected_components(augmented_graph)
    shared_counts = dynamics_feedback_context.H[:, dynamics_feedback_support].sum(axis=1)
    new_four_cycles = int(sum((math.comb(int(count), 2) for count in shared_counts if count >= 2)))
    structure_rows.append({
        'sample_id': dynamics_feedback_row.sample_id,
        'failure_type': dynamics_feedback_row.failure_type,
        'candidate_id': dynamics_feedback_row.candidate_id,
        'row_weight': dynamics_feedback_row.row_weight,
        'unstable_variables_covered': dynamics_feedback_row.overlap_count,
        'weighted_instability_overlap': dynamics_feedback_row.weighted_instability_overlap,
        'conflict_overlap': dynamics_feedback_row.conflict_overlap,
        'original_rows_combined': 2,
        'local_degree_change': dynamics_feedback_row.row_weight,
        'local_cycle_rank_change': after_rank - before_rank,
        'new_length_4_cycle_count': new_four_cycles,
        'raw_trajectory_gain': dynamics_feedback_row.raw_trajectory_gain,
        'syndrome_progress_gain': dynamics_feedback_row.syndrome_progress_gain,
        'repair': dynamics_feedback_row.repair,
        'regression': dynamics_feedback_row.regression,
    })

candidate_structure = pd.DataFrame(structure_rows)
candidate_structure_analysis = candidate_structure[candidate_structure.failure_type != 'not_failure'].groupby('repair', as_index=False).agg(
    candidates=('candidate_id', 'size'),
    samples=('sample_id', 'nunique'),
    mean_row_weight=('row_weight', 'mean'),
    mean_unstable_variables_covered=('unstable_variables_covered', 'mean'),
    mean_weighted_instability_overlap=('weighted_instability_overlap', 'mean'),
    mean_conflict_overlap=('conflict_overlap', 'mean'),
    mean_local_cycle_rank_change=('local_cycle_rank_change', 'mean'),
    mean_new_length_4_cycle_count=('new_length_4_cycle_count', 'mean'),
    mean_raw_gain=('raw_trajectory_gain', 'mean'),
)
candidate_structure_analysis.to_csv(dynamics_feedback_data_paths.output('candidate_structure_analysis_csv'), index=False)
dynamics_feedback_failure_rows = []

for dynamics_feedback_failure_type, dynamics_feedback_group in raw_primary.groupby('failure_type', sort=True):
    for dynamics_feedback_metric in RAW_METRICS:
        dynamics_feedback_gain = f'{dynamics_feedback_metric}_gain'
        auroc = roc_auc_score(dynamics_feedback_group.repair, dynamics_feedback_group[dynamics_feedback_gain]) if dynamics_feedback_group.repair.nunique() == 2 else np.nan
        dynamics_feedback_failure_rows.append({
            'failure_type': dynamics_feedback_failure_type,
            'metric': dynamics_feedback_metric,
            'candidates': len(dynamics_feedback_group),
            'samples': dynamics_feedback_group.sample_id.nunique(),
            'repair_rate': dynamics_feedback_group.repair.mean(),
            'regression_rate': dynamics_feedback_group.regression.mean(),
            'mean_raw_gain': dynamics_feedback_group[dynamics_feedback_gain].mean(),
            'mean_syndrome_progress_gain': dynamics_feedback_group.syndrome_progress_gain.mean(),
            'candidate_repair_AUROC': auroc,
        })

failure_type_mechanism = pd.DataFrame(dynamics_feedback_failure_rows)
failure_type_mechanism.to_csv(dynamics_feedback_data_paths.output('failure_type_mechanism_csv'), index=False)
candidate_structure_analysis


## Experiment N — Cross-p and cross-distance validation

Repeat the configured policy comparisons across p and distance. Results remain conditional on the achieved cohorts and tested action budgets.


In [ ]:
def deterministic_cross_distance_cases(level, context, probabilities):
    rows = []
    rng = np.random.default_rng(NEXT_CONFIG['next_random_seed'] + level * 100003)
    for probability in probabilities:
        _, probability_context = dynamics_feedback_context_for(level, float(probability))
        for weight in NEXT_CONFIG['cross_distance_error_weights']:
            total = min(NEXT_CONFIG['cross_distance_cases_per_stratum'], math.comb(probability_context.H.shape[1], weight))
            seen = set()
            while len(seen) < total:
                seen.add(tuple(sorted(rng.choice(probability_context.H.shape[1], weight, replace=False))))

            for support in sorted(seen):
                error = np.zeros(probability_context.H.shape[1], dtype=np.uint8)
                error[list(support)] = 1
                rows.append((probability, weight, error, probability_context))

    return rows

probability_grid = np.sort(dynamics_feedback_population_cases.p.unique())
grid_indices = np.linspace(0, len(probability_grid) - 1, min(len(probability_grid), NEXT_CONFIG['cross_distance_p_limit'])).round().astype(int)
selected_probabilities = probability_grid[grid_indices]
dynamics_feedback_cross_rows = []

for dynamics_feedback_level in tqdm(NEXT_CONFIG['cross_distance_levels'], desc='Experiment N: distance', disable=NEXT_CONFIG['print_level'] == 1):
    dynamics_feedback_distance, dynamics_feedback_reference = dynamics_feedback_context_for(dynamics_feedback_level, float(selected_probabilities[0]))
    for dynamics_feedback_case_index, (probability, dynamics_feedback_weight, dynamics_feedback_error, dynamics_feedback_context) in enumerate(deterministic_cross_distance_cases(dynamics_feedback_level, dynamics_feedback_reference, selected_probabilities)):
        dynamics_feedback_syndrome = dynamics_feedback_context.H @ dynamics_feedback_error % 2
        dynamics_feedback_truth = dynamics_feedback_logical_label(dynamics_feedback_context, dynamics_feedback_error)
        dynamics_feedback_baseline = dynamics_feedback_trace_min_sum(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_config['benchmark_iterations'])
        dynamics_feedback_, base_valid, base_correct, dynamics_feedback_ = outcome_at_first_valid(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_truth, dynamics_feedback_baseline)
        base_success = base_valid and base_correct
        localization_started = perf_counter()
        selection_probe = dynamics_feedback_trace_min_sum(dynamics_feedback_context, dynamics_feedback_syndrome, NEXT_CONFIG['probe_iterations'])
        localization_runtime = perf_counter() - localization_started
        localization_edge_updates = NEXT_CONFIG['probe_iterations'] * int(dynamics_feedback_context.H.sum())
        metrics = raw_dynamics(dynamics_feedback_context, selection_probe, NEXT_CONFIG['probe_iterations'])
        dynamics_feedback_region = dynamics_feedback_top_region(metrics.rank_trajectory, min(NEXT_CONFIG['primary_top_k'], dynamics_feedback_context.H.shape[1]))
        dynamics_feedback_candidates = dynamics_feedback_generate_redundant_candidates(dynamics_feedback_context.H, NEXT_CONFIG['candidate_max_row_weight'])
        overlaps = np.asarray([dynamics_feedback_candidate_overlap(candidate, metrics.rank_trajectory, dynamics_feedback_region)[1] for candidate in dynamics_feedback_candidates])
        dynamics_feedback_screened = balanced_candidate_screen(dynamics_feedback_candidates, overlaps, NEXT_CONFIG['candidate_screen_limit'], NEXT_CONFIG['next_random_seed'] + dynamics_feedback_level * 10007 + dynamics_feedback_case_index)
        dynamics_feedback_before = {value: raw_dynamics(dynamics_feedback_context, selection_probe, value) for value in NEXT_CONFIG['candidate_test_iterations']}
        evaluated = []

        for dynamics_feedback_candidate in dynamics_feedback_screened:
            dynamics_feedback_probes = probe_raw_candidate(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_candidate, dynamics_feedback_region, dynamics_feedback_before, selection_probe)
            dynamics_feedback_primary = [item for item in dynamics_feedback_probes if item['test_iterations'] == NEXT_CONFIG['primary_candidate_test_iterations']][0]
            evaluated.append((dynamics_feedback_candidate, dynamics_feedback_primary, overlaps[dynamics_feedback_candidate.candidate_id]))

        dynamics_feedback_gain_choice = max(evaluated, key=lambda item: (item[1]['raw_trajectory_gain'], -item[0].candidate_id))
        dynamics_feedback_overlap_choice = max(evaluated, key=lambda item: (item[2], -item[0].candidate_id))
        gains = minmax_within_group([item[1]['raw_trajectory_gain'] for item in evaluated])
        ov = minmax_within_group([item[2] for item in evaluated])
        hybrid_index = int(np.argmax(selected_alpha * gains + (1 - selected_alpha) * ov))
        dynamics_feedback_choices = [('overlap_targeted', dynamics_feedback_overlap_choice), ('raw_gain_targeted', dynamics_feedback_gain_choice), ('hybrid_targeted', evaluated[hybrid_index])]
        matched = [item for item in evaluated if item[0].row_weight == dynamics_feedback_gain_choice[0].row_weight]

        for dynamics_feedback_seed_index in range(NEXT_CONFIG['random_control_seeds']):
            dynamics_feedback_rng = np.random.default_rng(NEXT_CONFIG['next_random_seed'] + dynamics_feedback_level * 1000003 + dynamics_feedback_case_index * 101 + dynamics_feedback_seed_index)
            dynamics_feedback_choices.append(('random', evaluated[int(dynamics_feedback_rng.integers(len(evaluated)))]))
            dynamics_feedback_choices.append(('weight_matched_random', matched[int(dynamics_feedback_rng.integers(len(matched)))]))

        full_cache = {}
        for dynamics_feedback_strategy, (dynamics_feedback_candidate, dynamics_feedback_primary, overlap_score) in dynamics_feedback_choices:
            if dynamics_feedback_candidate.candidate_id not in full_cache:
                full_cache[dynamics_feedback_candidate.candidate_id] = full_raw_candidate(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_truth, dynamics_feedback_candidate, dynamics_feedback_region)
            dynamics_feedback_full = full_cache[dynamics_feedback_candidate.candidate_id]
            dynamics_feedback_success = bool(dynamics_feedback_full['syndrome_consistent'] and dynamics_feedback_full['logical_correct'])
            dynamics_feedback_cross_rows.append({
                'distance': dynamics_feedback_distance,
                'p': probability,
                'error_weight': dynamics_feedback_weight,
                'case_index': dynamics_feedback_case_index,
                'strategy': dynamics_feedback_strategy,
                'candidate_id': dynamics_feedback_candidate.candidate_id,
                'raw_gain': dynamics_feedback_primary['raw_trajectory_gain'],
                'overlap_score': overlap_score,
                'baseline_success': int(base_success),
                'repair': int(not base_success and dynamics_feedback_success),
                'regression': int(base_success and (not dynamics_feedback_success)),
                'syndrome_consistent': dynamics_feedback_full['syndrome_consistent'],
                'logical_correct': dynamics_feedback_full['logical_correct'],
                'selection_edge_updates': localization_edge_updates + sum((item[1]['probe_edge_updates'] for item in evaluated)),
                'decode_edge_updates': dynamics_feedback_full['full_edge_updates'],
                'total_edge_updates': localization_edge_updates + sum((item[1]['probe_edge_updates'] for item in evaluated)) + dynamics_feedback_full['full_edge_updates'],
                'selection_runtime_seconds': localization_runtime + sum((item[1]['probe_runtime_seconds'] for item in evaluated)),
                'decode_runtime_seconds': dynamics_feedback_full['full_runtime_seconds'],
                'total_runtime_seconds': localization_runtime + sum((item[1]['probe_runtime_seconds'] for item in evaluated)) + dynamics_feedback_full['full_runtime_seconds'],
            })

cross_distance_trials = pd.DataFrame(dynamics_feedback_cross_rows)
cross_distance_trials.to_csv(dynamics_feedback_data_paths.output('cross_distance_trials_csv'), index=False)
dynamics_feedback_cross_distance_summary = dynamics_feedback_conditional_outcome_rates(cross_distance_trials.groupby(['distance', 'p', 'strategy'], as_index=False).agg(
    evaluations=('case_index', 'size'),
    unique_cases=('case_index', 'nunique'),
    repair_count=('repair', 'sum'),
    regression_count=('regression', 'sum'),
    baseline_failure_evaluations=('baseline_success', lambda values: np.sum(values == 0)),
    baseline_success_evaluations=('baseline_success', lambda values: np.sum(values == 1)),
    mean_raw_instability_reduction=('raw_gain', 'mean'),
    mean_edge_updates=('total_edge_updates', 'mean'),
    median_runtime_seconds=('total_runtime_seconds', 'median'),
))
dynamics_feedback_cross_distance_summary['net_gain_rate'] = dynamics_feedback_cross_distance_summary.repair_rate - dynamics_feedback_cross_distance_summary.regression_rate

dynamics_feedback_cross_distance_summary.to_csv(dynamics_feedback_data_paths.output('raw_gain_cross_distance_csv'), index=False)
dynamics_feedback_cross_distance_summary


## Experiment O — Final full stochastic benchmark

Evaluate the complete feedback decoder on the physical-shot population. Compare whole-population LER with its reference, including failures introduced on baseline successes.


In [ ]:
def circuit_and_context(level, probability):
    code_object = CA_9_2_5() if level == 2 else TriangularCode(level)
    circuit = CircuitBuilder(code_object).get_circuit_level_circuit(ErrorRate(p=float(probability), p_meas=0), d_rounds=1)
    distance, context = dynamics_feedback_context_for(level, float(probability))

    return (distance, circuit, context)

def native_decode(context, syndrome, iterations):
    decoder = BpDecoder(
        context.H,
        error_channel=context.priors,
        max_iter=iterations,
        bp_method='minimum_sum',
        ms_scaling_factor=dynamics_feedback_config['ms_scaling_factor'],
        schedule='parallel',
        input_vector_type='syndrome',
    )
    started = perf_counter()
    correction = np.asarray(decoder.decode(syndrome), dtype=np.uint8)

    return (correction, int(decoder.iter), perf_counter() - started)

def bposd_decode(context, syndrome):
    decoder = BpOsdDecoder(
        context.H,
        error_channel=list(context.priors),
        max_iter=dynamics_feedback_config['benchmark_iterations'],
        bp_method='minimum_sum',
        ms_scaling_factor=dynamics_feedback_config['ms_scaling_factor'],
        osd_method='osd_cs',
        osd_order=NEXT_CONFIG['osd_order'],
        schedule='parallel',
    )
    started = perf_counter()
    correction = np.asarray(decoder.decode(syndrome), dtype=np.uint8)

    return (correction, int(decoder.iter), perf_counter() - started)

def adaptive_decode(context, syndrome):
    decoder = AdaptiveBpDecoder(
        pcm=context.H,
        error_channel=list(context.priors),
        max_iter=dynamics_feedback_config['benchmark_iterations'],
        bp_method='minimum_sum',
        adaptive_k=NEXT_CONFIG['adaptive_k'],
        adaptive_max_views=NEXT_CONFIG['adaptive_max_views'],
    )
    started = perf_counter()
    correction = np.asarray(decoder.decode(syndrome), dtype=np.uint8)
    actual_edge_updates = int(decoder.iter * context.H.sum() + np.sum(decoder.view_iterations * decoder.view_edge_counts))

    return (correction, actual_edge_updates, perf_counter() - started)

def targeted_decode(
    context,
    syndrome,
    truth_unused,
    strategy,
    seed,
):
    started_total = perf_counter()
    current = context
    current_syndrome = np.asarray(syndrome, dtype=np.uint8)
    selection_updates = 0
    maximum_rounds = NEXT_CONFIG['iterative_max_rounds'] if strategy == 'hybrid_feedback' else 1

    for round_index in range(maximum_rounds):
        baseline = dynamics_feedback_trace_min_sum(current, current_syndrome, NEXT_CONFIG['probe_iterations'])
        selection_updates += NEXT_CONFIG['probe_iterations'] * int(current.H.sum())
        if baseline.first_valid_iteration is not None:
            index = baseline.first_valid_iteration
            return (baseline.hard[index], selection_updates, perf_counter() - started_total)
        metrics = raw_dynamics(current, baseline, NEXT_CONFIG['probe_iterations'])
        region = dynamics_feedback_top_region(metrics.rank_trajectory, min(NEXT_CONFIG['primary_top_k'], current.H.shape[1]))
        candidates = dynamics_feedback_generate_redundant_candidates(current.H, NEXT_CONFIG['candidate_max_row_weight'])
        overlaps = np.asarray([dynamics_feedback_candidate_overlap(candidate, metrics.rank_trajectory, region)[1] for candidate in candidates])
        order = np.lexsort((np.arange(len(candidates)), -overlaps))[:selected_candidate_budget]
        screened = [candidates[index] for index in order]

        if strategy == 'random_redundant_view':
            rng = np.random.default_rng(seed + round_index)
            selected = screened[int(rng.integers(len(screened)))]
        elif strategy == 'overlap_targeted':
            selected = screened[0]
        else:
            before = raw_dynamics(current, baseline, NEXT_CONFIG['primary_candidate_test_iterations'])
            measured = []
            for candidate in screened:
                augmented = dynamics_feedback_augmented_context(current, candidate.row)
                augmented_s = dynamics_feedback_augmented_syndrome(current_syndrome, candidate.source_rows)
                trace = dynamics_feedback_trace_min_sum(augmented, augmented_s, NEXT_CONFIG['primary_candidate_test_iterations'])
                after = raw_dynamics(augmented, trace, NEXT_CONFIG['primary_candidate_test_iterations'])
                gain = metric_region_mean(before, 'raw_trajectory', region) - metric_region_mean(after, 'raw_trajectory', region)
                measured.append((gain, candidate, overlaps[candidate.candidate_id]))
                selection_updates += NEXT_CONFIG['primary_candidate_test_iterations'] * int(augmented.H.sum())

            gains = minmax_within_group([item[0] for item in measured])
            normalized_overlap = minmax_within_group([item[2] for item in measured])
            selected_index = int(np.argmax(selected_alpha * gains + (1 - selected_alpha) * normalized_overlap))
            selected_gain, selected, _ = measured[selected_index]

            if selected_gain <= max(0.0, selected_gain_threshold):
                break

        current_syndrome = dynamics_feedback_augmented_syndrome(current_syndrome, selected.source_rows)
        current = dynamics_feedback_augmented_context(current, selected.row)

    trace = dynamics_feedback_trace_min_sum(current, current_syndrome, dynamics_feedback_config['benchmark_iterations'])
    index = trace.first_valid_iteration if trace.first_valid_iteration is not None else dynamics_feedback_config['benchmark_iterations']

    return (trace.hard[index], selection_updates + int(dynamics_feedback_config['benchmark_iterations'] * current.H.sum()), perf_counter() - started_total)

stochastic_rows = []
for dynamics_feedback_level in tqdm(NEXT_CONFIG['cross_distance_levels'], desc='Experiment O: distance', disable=NEXT_CONFIG['print_level'] == 1):
    for probability in selected_probabilities:
        dynamics_feedback_distance, circuit, dynamics_feedback_context = circuit_and_context(dynamics_feedback_level, probability)
        sampler = circuit.compile_detector_sampler(seed=NEXT_CONFIG['next_random_seed'] + dynamics_feedback_level * 1009 + int(np.searchsorted(selected_probabilities, probability)))
        remaining = NEXT_CONFIG['stochastic_shots_per_point']
        shot_offset = 0

        while remaining:
            batch = min(remaining, NEXT_CONFIG['stochastic_batch_size'])
            detections, observables = sampler.sample(batch, separate_observables=True)
            for local_index in range(batch):
                dynamics_feedback_syndrome = detections[local_index].astype(np.uint8)
                dynamics_feedback_truth = dynamics_feedback_bit_text(observables[local_index].astype(np.uint8))
                shot_id = shot_offset + local_index
                methods = {
                    'plain_BP': lambda: native_decode(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_config['benchmark_iterations']),
                    'longer_plain_BP': lambda: native_decode(dynamics_feedback_context, dynamics_feedback_syndrome, NEXT_CONFIG['longer_bp_iterations']),
                    'BPOSD': lambda: bposd_decode(dynamics_feedback_context, dynamics_feedback_syndrome),
                    'current_Adaptive_BP': lambda: adaptive_decode(dynamics_feedback_context, dynamics_feedback_syndrome),
                    'random_redundant_view_BP': lambda: targeted_decode(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_truth, 'random_redundant_view', NEXT_CONFIG['next_random_seed'] + shot_id),
                    'overlap_targeted_BP': lambda: targeted_decode(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_truth, 'overlap_targeted', 0),
                    'instability_guided_feedback_BP': lambda: targeted_decode(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_truth, 'hybrid_feedback', NEXT_CONFIG['next_random_seed'] + shot_id),
                }

                for method, run in methods.items():
                    dynamics_feedback_correction, updates_or_iterations, runtime = run()
                    dynamics_feedback_valid = np.array_equal(dynamics_feedback_context.H @ dynamics_feedback_correction % 2, dynamics_feedback_syndrome)
                    dynamics_feedback_correct = dynamics_feedback_logical_label(dynamics_feedback_context, dynamics_feedback_correction) == dynamics_feedback_truth

                    if method in {'plain_BP', 'longer_plain_BP', 'BPOSD'}:
                        edge_updates = int(updates_or_iterations * dynamics_feedback_context.H.sum())
                    else:
                        edge_updates = int(updates_or_iterations)
                    stochastic_rows.append({
                        'distance': dynamics_feedback_distance,
                        'p': probability,
                        'shot_id': shot_id,
                        'method': method,
                        'syndrome_consistent': int(dynamics_feedback_valid),
                        'logical_correct': int(dynamics_feedback_correct),
                        'logical_error': int(not (dynamics_feedback_valid and dynamics_feedback_correct)),
                        'edge_updates': edge_updates,
                        'runtime_seconds': runtime,
                    })

            remaining -= batch
            shot_offset += batch

final_stochastic_trials = pd.DataFrame(stochastic_rows)
plain_outcomes = final_stochastic_trials[final_stochastic_trials.method == 'plain_BP'][['distance', 'p', 'shot_id', 'logical_error']].rename(columns={'logical_error': 'plain_logical_error'})
final_stochastic_trials = final_stochastic_trials.merge(plain_outcomes, on=['distance', 'p', 'shot_id'], validate='many_to_one')
final_stochastic_trials['repair'] = ((final_stochastic_trials.plain_logical_error == 1) & (final_stochastic_trials.logical_error == 0)).astype(int)
final_stochastic_trials['regression'] = ((final_stochastic_trials.plain_logical_error == 0) & (final_stochastic_trials.logical_error == 1)).astype(int)

final_stochastic_trials.to_csv(dynamics_feedback_data_paths.output('final_stochastic_trials_csv'), index=False)
final_stochastic_benchmark = final_stochastic_trials.groupby(['distance', 'p', 'method'], as_index=False).agg(
    shots=('shot_id', 'size'),
    LER=('logical_error', 'mean'),
    syndrome_invalid_rate=('syndrome_consistent', lambda values: 1 - values.mean()),
    repair_count=('repair', 'sum'),
    regression_count=('regression', 'sum'),
    baseline_failure_evaluations=('plain_logical_error', 'sum'),
    baseline_success_evaluations=('plain_logical_error', lambda values: np.sum(values == 0)),
    mean_edge_updates=('edge_updates', 'mean'),
    median_runtime_seconds=('runtime_seconds', 'median'),
)
final_stochastic_benchmark = dynamics_feedback_conditional_outcome_rates(final_stochastic_benchmark)

final_stochastic_benchmark.to_csv(dynamics_feedback_data_paths.output('final_stochastic_benchmark_csv'), index=False)
final_stochastic_benchmark


In [ ]:
final_stochastic_benchmark[(final_stochastic_benchmark['p'] >= 0.023101) & (final_stochastic_benchmark['p'] < 0.1)]


## Next-stage validation, figures, and interpretation guardrails


In [ ]:
validation_rows = []
for dynamics_feedback_sample in next_cases.head(NEXT_CONFIG['native_validation_cases']).itertuples(index=False):
    dynamics_feedback_, dynamics_feedback_context = dynamics_feedback_context_for(2, dynamics_feedback_sample.p)
    dynamics_feedback_syndrome = dynamics_feedback_bit_array(dynamics_feedback_sample.syndrome)
    instrumented = dynamics_feedback_trace_min_sum(dynamics_feedback_context, dynamics_feedback_syndrome, dynamics_feedback_config['benchmark_iterations'])
    dynamics_feedback_native = BpDecoder(
        dynamics_feedback_context.H,
        error_channel=dynamics_feedback_context.priors,
        max_iter=dynamics_feedback_config['benchmark_iterations'],
        bp_method='minimum_sum',
        ms_scaling_factor=dynamics_feedback_config['ms_scaling_factor'],
        schedule='parallel',
        input_vector_type='syndrome',
    )
    native_correction = np.asarray(dynamics_feedback_native.decode(dynamics_feedback_syndrome), dtype=np.uint8)
    dynamics_feedback_iteration = int(dynamics_feedback_native.iter)
    dynamics_feedback_native_llr = np.asarray(dynamics_feedback_native.log_prob_ratios)
    maximum_difference = float(np.max(np.abs(instrumented.llr[dynamics_feedback_iteration] - dynamics_feedback_native_llr)))
    assert maximum_difference <= NEXT_CONFIG['native_validation_tolerance']
    dynamics_feedback_hard_difference = instrumented.hard[dynamics_feedback_iteration] != native_correction

    if dynamics_feedback_hard_difference.any():
        assert np.max(np.abs(dynamics_feedback_native_llr[dynamics_feedback_hard_difference])) <= NEXT_CONFIG['native_validation_tolerance']
    validation_rows.append({
        'sample_id': dynamics_feedback_sample.sample_id,
        'iteration': dynamics_feedback_iteration,
        'hard_equal': int(not dynamics_feedback_hard_difference.any()),
        'valid_equal': int(bool(dynamics_feedback_native.converge) == bool(instrumented.syndrome_consistent[dynamics_feedback_iteration])),
        'maximum_posterior_difference': maximum_difference,
    })

next_native_validation = pd.DataFrame(validation_rows)
next_native_validation.to_csv(dynamics_feedback_data_paths.output('next_native_validation_csv'), index=False)
assert next_native_validation.valid_equal.all()
expected_stats = {'raw_gain_vs_repair.csv', 'instability_metric_ablation.csv', 'overlap_gain_joint_analysis.csv', 'hybrid_targeting_ablation.csv', 'unbiased_candidate_validation.csv', 'trajectory_redirection_analysis.csv', 'early_intervention_stable_wrong.csv', 'fixed_region_feedback_summary.csv', 'intervention_gating.csv', 'candidate_budget_ablation.csv', 'candidate_structure_analysis.csv', 'failure_type_mechanism.csv', 'raw_gain_cross_distance.csv', 'final_stochastic_benchmark.csv', 'next_instability_input_audit.csv', 'feedback_stopping_rules.csv', 'next_native_validation.csv', 'final_decoder_development_tuning.csv'}
missing = expected_stats - {path.name for path in NEXT_STATS.glob('*.csv')}
assert not missing, missing

dynamics_feedback_fig, dynamics_feedback_axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
quintiles = failure_candidates.copy()
quintiles['gain_quintile'] = pd.qcut(quintiles.raw_trajectory_gain.rank(method='first'), 5, labels=['0-20', '20-40', '40-60', '60-80', '80-100'])
sns.barplot(
    data=quintiles,
    x='gain_quintile',
    y='repair',
    ax=dynamics_feedback_axes[0, 0],
)
dynamics_feedback_axes[0, 0].set_title('Raw trajectory gain vs repair')
sns.scatterplot(
    data=repair_frame.sample(min(5000, len(repair_frame)), random_state=NEXT_CONFIG['next_random_seed']),
    x='weighted_instability_overlap',
    y='raw_trajectory_gain',
    hue='repair',
    alpha=0.5,
    ax=dynamics_feedback_axes[0, 1],
)
dynamics_feedback_axes[0, 1].set_title('Spatial overlap vs dynamical gain')
sns.barplot(
    data=instability_metric_ablation[instability_metric_ablation.task == 'repair'],
    x='metric',
    y='AUROC',
    ax=dynamics_feedback_axes[0, 2],
)
dynamics_feedback_axes[0, 2].tick_params(axis='x', rotation=45)
dynamics_feedback_axes[0, 2].set_title('Metric ablation')
sns.lineplot(
    data=early_intervention_summary,
    x='probe_iterations',
    y='repair_rate',
    hue='strategy',
    marker='o',
    ax=dynamics_feedback_axes[1, 0],
)
dynamics_feedback_axes[1, 0].set_title('Early intervention')
sns.lineplot(
    data=iterative_feedback_trajectories,
    x='round',
    y='fixed_region_instability',
    hue='failure_type',
    estimator='median',
    ax=dynamics_feedback_axes[1, 1],
)
dynamics_feedback_axes[1, 1].set_title('Fixed-region feedback')
sns.scatterplot(
    data=candidate_budget_ablation,
    x='mean_edge_updates',
    y='repair_rate',
    hue='search',
    ax=dynamics_feedback_axes[1, 2],
)
dynamics_feedback_axes[1, 2].set_title('Accuracy-cost frontier')

plt.show()
next_log(2, {
    'selected_hybrid_alpha': selected_alpha,
    'raw_gain_repair_AUROC': raw_gain_vs_repair.loc[raw_gain_vs_repair.metric == 'raw_trajectory', 'repair_AUROC'].iloc[0],
    'stats_files': len(list(NEXT_STATS.glob('*.csv'))),
    'trace_files': len(list(NEXT_TRACE.glob('*.csv'))),
})


### Interpretation guardrails

- Raw before/after quantities use the same original variable region and iteration window.
- Candidate selection never uses the evaluated logical outcome.
- Candidate-level predictive models use `GroupKFold(sample_id)`.
- Hybrid, gating, stopping, and final-decoder tuning use only the development split;
  reported comparisons use held-out samples.
- Reduced instability without syndrome progress is trajectory suppression, not evidence
  of useful redirection.
- Cross-distance deterministic-error experiments are mechanism checks, not population LER.
- Final stochastic `LER` is `not (syndrome_consistent and logical_correct)`.
- Trace-like candidate, iteration, and shot records are stored only under `trace/`.
